# Simulation, part 2: published model, WHO BMI-group model and effect-size grid

Same cohort geometry, truth and data generator as the completed Student-t simulation (`k1_worker.py`, study-config hash checked).
1. **Published model** (exact `bhm_extension.py` legacy code: patient effect orthogonal to intercept, BMI, age and sex) on the **same 120 datasets** (6 scenarios × 20).
2. **WHO BMI-group model** (Student-t patient × cell-type intercept, as fitted to the real data) and **OLS by group ± BH** on the same 120 datasets. Truth for each contrast: the simulated BMI effect × the group's mean BMI (z) difference from normal weight in that dataset.
3. **Effect-size grid**: cell-type BMI effect 0.05, 0.10, 0.20, 0.30 SD per SD of BMI (fitted between-feature spread), 20 replicates each: Student-t, published model, OLS ± BH, for power curves.

In [ ]:
# 1. Install the validated sampling environment
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pymc==5.28.5", "arviz==0.23.4", "nutpie==0.16.11"])
print("Pinned environment installed.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
assert Path('/content/drive/MyDrive/new_dysfunction').is_dir(), (
    'MyDrive/new_dysfunction not visible. Confirm correct Google account and Drive mount.'
)
print('Drive mounted and study input root visible.')


In [ ]:
%%writefile /content/ri_hc_engine.py
FAMILY = CONFIG["family"]
PROFILE = CONFIG["profile"]
STAGE = CONFIG["stage"]
MIN_READS = CONFIG.get("min_reads", 1000)
LOCAL_PATHS = CONFIG.get("local_paths")
SAMPLER_OVERRIDE = CONFIG.get("sampler_override")
PPC_DRAWS_OVERRIDE = CONFIG.get("ppc_draws_override")
PROMOTION_WAIVER = CONFIG.get("waiver")

import gc, hashlib, json, math, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "ri-hc-celltype-intercept-v1"
SCORE_VARIANT = "within_component_logcpm_z_cap5"
BASE = Path("/content/drive/MyDrive/new_dysfunction")
PCA_DIR = BASE / ("bayesprism_initial_issue_verification_v1/publication_pca_tiered_qc_v1/"
                  "20260906T013040524557Z")
SENS_ROOT = BASE / "normalized_state_bhm_ri_hc_v1"
PRIMARY_LOWREAD_ROOT = BASE / "normalized_state_bhm_lowread_sensitivity_v1"
CELLTYPE_INTERCEPT_SD = 0.50
WIDENING = 1.37
HEADLINE = {
    "immune_coarse": [("CD4+ T CELLS", "Helper_Function_Signature"),
                      ("CD4+ T CELLS", "CD4_Metabolic_Flexibility_Signature"),
                      ("CD8_T_GENERAL", "CD8_Metabolic_Programming_Signature")],
    "immune_fine": [("TAM", "Iron_Handling_Signature"), ("TAM", "Lipid_Processing_Signature"),
                    ("TAM", "Tissue_Repair_Signature"), ("CD8_T_EXHAUSTED", "Metabolic_Imbalance_Signature")],
    "nonimmune": [],
}

FAMILY_SPECS = {
    "immune_coarse": {
        "compartment": "immune_course", "tier_column": "tiered_primary_keep",
        "nu_scope": "celltype", "architecture": "v2p2_g1_fixed_ctnu",
        "role": "primary", "expected_features": 306, "expected_celltypes": 8,
        "expected_dropped_cells": 25,
        "original_run": BASE / "normalized_state_bhm_v2p2_celltype_nu" / (
            "v2p2_g1_fixed_ctnu__immune_coarse_primary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "immune_coarse_v2p2_feature_bmi_posterior.csv",
    },
    "nonimmune": {
        "compartment": "non_immune", "tier_column": "strict_qc_keep",
        "nu_scope": "global", "architecture": "v2p1c_g1_fixed_gnu",
        "role": "primary", "expected_features": 112, "expected_celltypes": 4,
        "expected_dropped_cells": 2,
        "original_run": BASE / "normalized_state_bhm_v2p1c_separate_compartments" / (
            "v2p1c_g1_fixed_gnu__nonimmune_primary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "nonimmune_v2p1c_feature_bmi_posterior.csv",
    },
    "immune_fine": {
        "compartment": "immune_fine", "tier_column": "tiered_primary_keep",
        "nu_scope": "global", "architecture": "v2p1c_g1_fixed_gnu",
        "role": "secondary", "expected_features": 496, "expected_celltypes": 14,
        "expected_dropped_cells": 58,
        "original_run": BASE / "normalized_state_bhm_v2p1c_separate_compartments" / (
            "v2p1c_g1_fixed_gnu__immune_fine_secondary__within_component_logcpm_z_cap5"
            "__all__linear__theta_0__publication"),
        "original_table": "immune_fine_v2p1c_feature_bmi_posterior.csv",
    },
}

PROFILES = {
    "smoke": {"tune": 500, "draws": 500, "chains": 2, "target_accept": 0.97},
    "standard": {"tune": 1000, "draws": 1000, "chains": 4, "target_accept": 0.98},
    "publication": {"tune": 2000, "draws": 4000, "chains": 4, "target_accept": 0.99},
}
PREDECESSOR = {"smoke": None, "standard": "smoke", "publication": "standard"}
PPC_DRAWS = {"smoke": 200, "standard": 2000, "publication": 4000}

BASELINE_SD = 0.50
CELLTYPE_BMI_SD = 0.20
FEATURE_BMI_SD = 0.30
CELLTYPE_COVARIATE_SD = 0.25
FEATURE_COVARIATE_SD = 0.25
PATIENT_GLOBAL_SD = 0.50
LOG_SIGMA_LOCATION = math.log(0.65)
LOG_SIGMA_LOCATION_SD = 0.40
CELLTYPE_LOG_SIGMA_SD = 0.25
FEATURE_LOG_SIGMA_SD = 0.50
NU_MINUS_TWO_RATE = 0.10

PRIMARY_ROPE = 0.10
ROPE_GRID = (0.05, 0.10, 0.15, 0.20)
Q_LEVEL = 0.05
MAX_TREE_DEPTH = int(CONFIG.get("max_tree_depth", 10))
MAX_TREE_DEPTH_FRACTION = 0.01
BFMI_MIN = 0.30

SMOKE_RHAT_HARD_MAX = 1.10
GATES = {
    "smoke": {"rhat_max": 1.05, "ess_min": 100},
    "standard": {"rhat_max": 1.01, "ess_min": 400},
    "publication": {"rhat_max": 1.01, "ess_min": 400},
}
PPC_ALPHA = 0.025
PPC_CORRELATION_THRESHOLD = 0.90
PPC_BATCH = 32

LOCKED_HASHES = {
    "pca_manifest.json": "a4263b46efb1c0d1d98a01b127ddb143bb7616ce88f8b364959613bbb74dcd30",
    "component_inclusion_tiers.csv": "16494a18ed467223134f0aed2efbf63c084d2f6b09332ca9854a9c403dd67294",
    "pca_input_hashes.csv": "6e2eaedbb5efe7c00d82e82c939d4416a5a3853f14bde881a55471cf874259c9",
    "combined_review_manifest.json": "04b0b1d788fc69e78ded4b44e050d487bfe231dfd5e5bd2b34f6b359c74f4476",
}
DECISION_RULE = {
    "any_bfdr_selection_change": True,
    "sign_flip_among_hdi_excluding_zero_features": True,
    "hdi_status_change_fraction_max": 0.10,
    "hdi_status_change_count_min": 3,
    "celltype_mean_slope_shift_max": 0.05,
}

if FAMILY not in FAMILY_SPECS:
    raise ValueError(f"FAMILY must be one of {sorted(FAMILY_SPECS)}")
if PROFILE not in PROFILES:
    raise ValueError(f"PROFILE must be one of {sorted(PROFILES)}")
if STAGE not in {"preflight", "fit", "diagnostics", "ppc", "compare"}:
    raise ValueError("Unknown STAGE")
if STAGE == "compare" and PROFILE != "publication":
    raise ValueError("compare uses the publication fit; set PROFILE = 'publication'")
if MIN_READS != 1000:
    warnings.warn("MIN_READS differs from the prespecified 1,000-read threshold.")

SPEC = FAMILY_SPECS[FAMILY]
PRIMARY_RUN_NAME = f"{SPEC['architecture']}__{FAMILY}__lowread{MIN_READS}"
RUN_NAME = f"{PRIMARY_RUN_NAME}__ri_hc" + ("" if MAX_TREE_DEPTH == 10 else f"_depth{MAX_TREE_DEPTH}")
RUN_SEED = 20260916 + int(hashlib.sha256(RUN_NAME.encode()).hexdigest()[:8], 16) % 1_000_000
ORIGINAL_RUN = Path(SPEC["original_run"])
if LOCAL_PATHS:
    PCA_DIR = Path(LOCAL_PATHS["pca_dir"])
    SENS_ROOT = Path(LOCAL_PATHS["sens_root"])
    ORIGINAL_RUN = Path(LOCAL_PATHS["original_runs"][FAMILY])
RUN_ROOT = SENS_ROOT / RUN_NAME
OUTPUT_DIR = RUN_ROOT / PROFILE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRACE_PATH = OUTPUT_DIR / "posterior_trace.nc"
print(f"{RUN_NAME} | profile {PROFILE} | stage {STAGE} | seed {RUN_SEED}")
print("Output:", OUTPUT_DIR)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

def write_json(value, path):
    tmp = Path(path).with_suffix(".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def write_csv(frame, path):
    tmp = Path(str(path) + ".tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(path)

def _clean_attrs(attrs):
    for key in list(attrs):
        value = attrs[key]
        if value is None or isinstance(value, (dict, set, bool)) or (
                isinstance(value, (list, tuple)) and any(isinstance(v, (dict, list, tuple, type(None))) for v in value)):
            attrs[key] = json.dumps(value, default=str, sort_keys=True)

def sanitize_attrs(idata):
    """netCDF rejects nested metadata (nutpie stores a dict); store it as JSON text."""
    _clean_attrs(idata.attrs)
    for group in idata.groups():
        _clean_attrs(idata[group].attrs)

def strict_bool(series):
    mapped = series.map({True: True, False: False, "True": True, "False": False,
                         "true": True, "false": False})
    if mapped.isna().any():
        raise ValueError("Non-boolean inclusion flag")
    return mapped.astype(bool)

# 4. Load the frozen inputs and apply the low-read rule
for name, expected in LOCKED_HASHES.items():
    folder = PCA_DIR
    if name == "combined_review_manifest.json":
        continue
    actual = sha256_file(folder / name)
    if actual != expected:
        raise RuntimeError(f"{name} does not match the locked hash")

pca_manifest = json.loads((PCA_DIR / "pca_manifest.json").read_text())
REVIEW_DIR = Path(pca_manifest["combined_review_directory"])
if LOCAL_PATHS:
    REVIEW_DIR = Path(LOCAL_PATHS["review_dir"])
review_manifest_path = REVIEW_DIR / "combined_review_manifest.json"
if sha256_file(review_manifest_path) != LOCKED_HASHES["combined_review_manifest.json"]:
    raise RuntimeError("combined_review_manifest.json does not match the locked hash")
review_manifest = json.loads(review_manifest_path.read_text())
RUN_DIR = Path(review_manifest["selected_runs"][SPEC["compartment"]])
if LOCAL_PATHS:
    RUN_DIR = Path(LOCAL_PATHS["run_dirs"][SPEC["compartment"]])

input_hashes = pd.read_csv(PCA_DIR / "pca_input_hashes.csv")
expected_input = input_hashes[input_hashes["compartment"].eq(SPEC["compartment"])]
cohort_path = RUN_DIR / "analysis_cohort.csv"
score_path = RUN_DIR / "audit_scores_all_variants.csv.gz"
qc_path = RUN_DIR / "patient_qc.csv"
for kind, path in (("cohort", cohort_path), ("scores", score_path)):
    locked = expected_input.loc[expected_input["input"].eq(kind), "sha256"]
    if len(locked) != 1 or sha256_file(path) != locked.iloc[0]:
        raise RuntimeError(f"{path.name} does not match pca_input_hashes.csv")

cohort = pd.read_csv(cohort_path, dtype={"base_sample_id": str})
scores = pd.read_csv(score_path, dtype={"base_sample_id": str})
patient_qc = pd.read_csv(qc_path, dtype={"base_sample_id": str})
tiers = pd.read_csv(PCA_DIR / "component_inclusion_tiers.csv")

if cohort["base_sample_id"].nunique() != 127:
    raise ValueError("Expected 127 patients")
if cohort["bmi_group"].value_counts().to_dict() != {"overweight": 57, "normal_weight": 52, "obese": 18}:
    raise ValueError("BMI group counts changed")

local_tiers = tiers[tiers["compartment"].eq(SPEC["compartment"])]
keep_celltypes = sorted(local_tiers.loc[strict_bool(local_tiers[SPEC["tier_column"]]), "CellType"].astype(str))
covariate_columns = ["base_sample_id", "bmi", "bmi_z", "bmi_group", "age_z", "sex_centered"]
frame = scores[scores["variant"].eq(SCORE_VARIANT) & scores["CellType"].isin(keep_celltypes)].copy()
frame["global_celltype"] = SPEC["compartment"] + "||" + frame["CellType"].astype(str)
frame["global_feature"] = SPEC["compartment"] + "||" + frame["feature"].astype(str)
frame = frame.merge(cohort[covariate_columns], on="base_sample_id", how="inner", validate="many_to_one")
if frame.duplicated(["base_sample_id", "global_feature"]).any():
    raise ValueError("Duplicate patient/feature rows")

frame["score"] = pd.to_numeric(frame["score"], errors="coerce")
grouped = frame.groupby("global_feature")["score"]
frame["score_model"] = (frame["score"] - grouped.transform("mean")) / grouped.transform("std")
frame["bmi_standardized"] = frame["bmi_z"]
if not np.isfinite(frame[["score_model", "bmi_standardized", "age_z", "sex_centered"]].to_numpy(float)).all():
    raise ValueError("Nonfinite model values")
if not frame.groupby("global_feature")["base_sample_id"].nunique().eq(127).all():
    raise ValueError("Every feature must contain all 127 patients before the low-read rule")

n_features = frame["global_feature"].nunique()
n_celltypes = frame["global_celltype"].nunique()
if (n_features, n_celltypes) != (SPEC["expected_features"], SPEC["expected_celltypes"]):
    raise ValueError(f"Expected {SPEC['expected_features']} features / {SPEC['expected_celltypes']} "
                     f"cell types, found {n_features} / {n_celltypes}")

original_inventory_path = ORIGINAL_RUN / "feature_inventory.csv"
if original_inventory_path.exists():
    original_features = set(pd.read_csv(original_inventory_path)["global_feature"])
    if original_features != set(frame["global_feature"]):
        raise ValueError("Feature set differs from the final fit's feature_inventory.csv")
    print("Feature set matches the final fit.")
else:
    warnings.warn(f"Cannot check against {original_inventory_path}")

reads = patient_qc[["base_sample_id", "CellType", "component_total", "theta"]].copy()
reads["component_total"] = pd.to_numeric(reads["component_total"], errors="coerce")
frame = frame.merge(reads, on=["base_sample_id", "CellType"], how="left", validate="many_to_one")
if frame["component_total"].isna().any():
    raise ValueError("Missing allocated-read totals")
frame["low_read"] = frame["component_total"] < MIN_READS

dropped_cells = (frame.loc[frame["low_read"], ["base_sample_id", "CellType", "component_total", "theta", "bmi"]]
                 .drop_duplicates(["base_sample_id", "CellType"])
                 .sort_values(["CellType", "component_total"]).reset_index(drop=True))
n_dropped_obs = int(frame["low_read"].sum())
if len(dropped_cells) != SPEC["expected_dropped_cells"] and MIN_READS == 1000:
    raise ValueError(f"Expected {SPEC['expected_dropped_cells']} low-read cells, found {len(dropped_cells)}")

full_frame = frame.copy()
analysis_data = frame.loc[~frame["low_read"]].reset_index(drop=True)
per_feature = analysis_data.groupby("global_feature")["base_sample_id"].nunique()
if per_feature.min() < 30:
    raise ValueError(f"A feature keeps only {per_feature.min()} patients")
if analysis_data["base_sample_id"].nunique() != 127:
    raise ValueError("A patient lost every observation")

write_csv(dropped_cells, RUN_ROOT / "dropped_low_read_cells.csv")
preflight = {
    "notebook_version": NOTEBOOK_VERSION, "family": FAMILY, "role": SPEC["role"],
    "architecture": SPEC["architecture"], "nu_scope": SPEC["nu_scope"],
    "min_reads": MIN_READS, "seed": RUN_SEED,
    "n_patients": 127, "n_features": n_features, "n_celltypes": n_celltypes,
    "n_observations_full": len(full_frame), "n_observations_fitted": len(analysis_data),
    "n_dropped_cells": len(dropped_cells), "n_dropped_observations": n_dropped_obs,
    "dropped_cells_by_celltype": dropped_cells["CellType"].value_counts().to_dict(),
    "minimum_patients_per_feature": int(per_feature.min()),
    "standardisation": "127-patient feature mean/SD from the final fits (before removal)",
    "input_sha256": {"cohort": sha256_file(cohort_path), "scores": sha256_file(score_path),
                     "patient_qc": sha256_file(qc_path)},
    "decision_rule": DECISION_RULE,
}
print(json.dumps(preflight, indent=2, default=str))
display_cells = dropped_cells.copy()
display_cells["component_total"] = display_cells["component_total"].round(1)
print(display_cells.to_string(index=False))

# 5. Build the model (final architecture, incomplete grid)
import pymc as pm
import pytensor.tensor as pt
import arviz as az

def prepare_arrays(data):
    data = data.sort_values(["global_celltype", "global_feature", "base_sample_id"]).reset_index(drop=True).copy()
    features = sorted(data["global_feature"].unique())
    celltypes = sorted(data["global_celltype"].unique())
    patients = sorted(full_frame["base_sample_id"].unique())
    f_index = {n: i for i, n in enumerate(features)}
    c_index = {n: i for i, n in enumerate(celltypes)}
    p_index = {n: i for i, n in enumerate(patients)}
    data["feature_idx"] = data["global_feature"].map(f_index)
    data["celltype_idx"] = data["global_celltype"].map(c_index)
    data["patient_idx"] = data["base_sample_id"].map(p_index)
    mapping = data[["feature_idx", "celltype_idx"]].drop_duplicates().sort_values("feature_idx")
    if len(mapping) != len(features):
        raise ValueError("Every feature must map to one cell type")
    feature_to_celltype = mapping["celltype_idx"].to_numpy(int)

    predictors = ["bmi_standardized", "age_z", "sex_centered"]
    table = (full_frame[["base_sample_id", *predictors]].drop_duplicates("base_sample_id")
             .set_index("base_sample_id").loc[patients])
    design = np.column_stack([np.ones(len(patients)), table.to_numpy(float)])
    if np.linalg.matrix_rank(design) != design.shape[1]:
        raise ValueError("Patient design is rank deficient")
    u, s, _ = np.linalg.svd(design, full_matrices=True)
    rank = int((s > max(design.shape) * np.finfo(float).eps * s.max()).sum())
    basis = u[:, rank:] * np.sqrt(len(patients) / (len(patients) - rank))
    if np.abs(design.T @ basis).max() > 1e-10:
        raise AssertionError("Patient basis is not orthogonal")
    return {"data": data, "features": features, "celltypes": celltypes, "patients": patients,
            "feature_to_celltype": feature_to_celltype, "patient_basis": basis,
            "design_rank": rank,
            "patient_covariates": table.to_numpy(float)}

def build_model(arrays):
    data = arrays["data"]
    covariates = ["age_z", "sex_centered"]
    coords = {"obs_id": np.arange(len(data)), "feature": arrays["features"],
              "celltype": arrays["celltypes"], "patient": arrays["patients"],
              "patient_re_dimension": np.arange(arrays["patient_basis"].shape[1]),
              "covariate": covariates}
    with pm.Model(coords=coords) as model:
        feature_idx = pm.Data("feature_idx", data["feature_idx"].to_numpy(int), dims="obs_id")
        patient_idx = pm.Data("patient_idx", data["patient_idx"].to_numpy(int), dims="obs_id")
        celltype_idx = pm.Data("celltype_idx", data["celltype_idx"].to_numpy(int), dims="obs_id")
        f2c = pm.Data("feature_to_celltype", arrays["feature_to_celltype"], dims="feature")
        bmi_z = pm.Data("bmi_z", data["bmi_standardized"].to_numpy(float), dims="obs_id")
        cov = pm.Data("covariate_matrix", data[covariates].to_numpy(float), dims=("obs_id", "covariate"))
        y = pm.Data("y", data["score_model"].to_numpy(float), dims="obs_id")
        patient_bmi = pm.Data("patient_bmi", arrays["patient_covariates"][:, 0], dims="patient")
        patient_cov = pm.Data("patient_covariate_matrix", arrays["patient_covariates"][:, 1:],
                              dims=("patient", "covariate"))

        feature_baseline = pm.Normal("feature_baseline", 0.0, BASELINE_SD, dims="feature")
        celltype_bmi_slope = pm.Normal("celltype_bmi_slope", 0.0, CELLTYPE_BMI_SD, dims="celltype")
        feature_sigma_bmi = pm.HalfNormal("feature_sigma_bmi", FEATURE_BMI_SD, dims="celltype")
        feature_bmi_raw = pm.Normal("feature_bmi_raw", 0.0, 1.0, dims="feature")
        feature_bmi_slope = pm.Deterministic(
            "feature_bmi_slope",
            celltype_bmi_slope[f2c] + feature_bmi_raw * feature_sigma_bmi[f2c], dims="feature")

        celltype_covariate_slope = pm.Normal("celltype_covariate_slope", 0.0, CELLTYPE_COVARIATE_SD,
                                             dims=("covariate", "celltype"))
        feature_sigma_covariate = pm.HalfNormal("feature_sigma_covariate", FEATURE_COVARIATE_SD,
                                                dims=("covariate", "celltype"))
        feature_covariate_raw = pm.Normal("feature_covariate_raw", 0.0, 1.0, dims=("covariate", "feature"))
        feature_covariate_slope = pm.Deterministic(
            "feature_covariate_slope",
            celltype_covariate_slope[:, f2c] + feature_covariate_raw * feature_sigma_covariate[:, f2c],
            dims=("covariate", "feature"))

        celltype_intercept_sigma = pm.HalfNormal("celltype_intercept_sigma", CELLTYPE_INTERCEPT_SD,
                                                 dims="celltype")
        w_mean = (patient_bmi[:, None] * celltype_bmi_slope[None, :]
                  + pt.dot(patient_cov, celltype_covariate_slope))
        celltype_intercept_w = pm.Normal("celltype_intercept_w", w_mean, celltype_intercept_sigma[None, :],
                                         dims=("patient", "celltype"))
        bmi_deviation = feature_bmi_raw * feature_sigma_bmi[f2c]
        covariate_deviation = feature_covariate_raw * feature_sigma_covariate[:, f2c]
        mu = (feature_baseline[feature_idx] + bmi_deviation[feature_idx] * bmi_z
              + celltype_intercept_w[patient_idx, celltype_idx]
              + pt.sum(cov * covariate_deviation[:, feature_idx].T, axis=1))

        log_sigma_location = pm.Normal("log_sigma_location", LOG_SIGMA_LOCATION, LOG_SIGMA_LOCATION_SD)
        log_sigma_celltype_location_sd = pm.HalfNormal("log_sigma_celltype_location_sd", CELLTYPE_LOG_SIGMA_SD)
        log_sigma_celltype_raw = pm.Normal("log_sigma_celltype_raw", 0.0, 1.0, dims="celltype")
        log_sigma_celltype_location = pm.Deterministic(
            "log_sigma_celltype_location",
            log_sigma_location + log_sigma_celltype_location_sd * log_sigma_celltype_raw, dims="celltype")
        log_sigma_feature_scale = pm.HalfNormal("log_sigma_feature_scale", FEATURE_LOG_SIGMA_SD, dims="celltype")
        log_sigma_raw = pm.Normal("log_sigma_raw", 0.0, 1.0, dims="feature")
        sigma_feature = pm.Deterministic(
            "sigma_feature",
            pt.exp(log_sigma_celltype_location[f2c] + log_sigma_feature_scale[f2c] * log_sigma_raw),
            dims="feature")

        if SPEC["nu_scope"] == "celltype":
            nu_minus_two = pm.Exponential("nu_minus_two", lam=NU_MINUS_TWO_RATE, dims="celltype")
            nu = pm.Deterministic("nu", nu_minus_two + 2.0, dims="celltype")
            nu_obs = nu[celltype_idx]
        else:
            nu_minus_two = pm.Exponential("nu_minus_two", lam=NU_MINUS_TWO_RATE)
            nu = pm.Deterministic("nu", nu_minus_two + 2.0)
            nu_obs = nu
        pm.StudentT("y_obs", nu=nu_obs, mu=mu, sigma=sigma_feature[feature_idx], observed=y, dims="obs_id")
    return model

arrays = prepare_arrays(analysis_data)
model = build_model(arrays)
initial_logp = float(model.compile_logp()(model.initial_point()))
if not np.isfinite(initial_logp):
    raise ValueError("Initial log density is not finite")
expected_rvs = {"feature_baseline", "celltype_bmi_slope", "feature_sigma_bmi", "feature_bmi_raw",
                "celltype_covariate_slope", "feature_sigma_covariate", "feature_covariate_raw",
                "celltype_intercept_sigma", "celltype_intercept_w", "log_sigma_location",
                "log_sigma_celltype_location_sd", "log_sigma_celltype_raw",
                "log_sigma_feature_scale", "log_sigma_raw", "nu_minus_two"}
if {rv.name for rv in model.free_RVs} != expected_rvs:
    raise AssertionError("Free random variables differ from the ri_hc architecture")
if (model["nu"].ndim == 1) != (SPEC["nu_scope"] == "celltype"):
    raise AssertionError("nu scope does not match the final architecture")

structure = {**preflight, "initial_logp": initial_logp,
             "patient_basis_dimensions": int(arrays["patient_basis"].shape[1]),
             "free_rvs": sorted(expected_rvs)}
write_json(structure, RUN_ROOT / "preflight_manifest.json")
print(f"Model built: {len(arrays['data']):,} observations, initial logp {initial_logp:.1f}")

# 6. Fit (with enforced promotion) and convergence gates
def gate_path(profile):
    return RUN_ROOT / profile / "run_diagnostics.json"

def compute_diagnostics(idata, profile):
    post = idata.posterior
    names = [v for v in post.data_vars]
    rhat = az.rhat(post[names])
    ess_bulk = az.ess(post[names], method="bulk")
    ess_tail = az.ess(post[names], method="tail")
    rows = []
    for name in names:
        r = np.atleast_1d(rhat[name].values).ravel()
        b = np.atleast_1d(ess_bulk[name].values).ravel()
        t = np.atleast_1d(ess_tail[name].values).ravel()
        rows.append({"parameter_family": name, "n": r.size, "max_rhat": float(np.nanmax(r)),
                     "min_ess_bulk": float(np.nanmin(b)), "min_ess_tail": float(np.nanmin(t)),
                     "nonfinite": int((~np.isfinite(r)).sum() + (~np.isfinite(b)).sum())})
    families = pd.DataFrame(rows).sort_values("max_rhat", ascending=False)
    stats = idata.sample_stats
    divergences = int(stats["diverging"].values.sum())
    bfmi = [float(x) for x in np.atleast_1d(az.bfmi(idata))]
    if "maxdepth_reached" in stats:
        maxdepth_fraction = stats["maxdepth_reached"].values.reshape(len(bfmi), -1).mean(axis=1)
    else:
        maxdepth_fraction = (stats["tree_depth"].values >= MAX_TREE_DEPTH).reshape(len(bfmi), -1).mean(axis=1)
    gate = GATES[profile]
    record = {
        "family": FAMILY, "profile": profile, "run_name": RUN_NAME,
        "chains": int(post.sizes["chain"]), "draws_per_chain": int(post.sizes["draw"]),
        "divergences": divergences, "min_bfmi": min(bfmi),
        "max_maxdepth_fraction": float(np.max(maxdepth_fraction)),
        "max_rhat": float(families["max_rhat"].max()),
        "min_ess_bulk": float(families["min_ess_bulk"].min()),
        "min_ess_tail": float(families["min_ess_tail"].min()),
        "nonfinite_diagnostics": int(families["nonfinite"].sum()),
        "trace_sha256": sha256_file(TRACE_PATH) if TRACE_PATH.exists() else None,
        "code_test_override": SAMPLER_OVERRIDE,
        "promotion_waiver": (json.loads((OUTPUT_DIR / "promotion_waiver.json").read_text())["reason"]
                             if (OUTPUT_DIR / "promotion_waiver.json").exists() else None),
    }
    record["gate_pass"] = bool(
        SAMPLER_OVERRIDE is None and divergences == 0 and min(bfmi) >= BFMI_MIN
        and record["max_maxdepth_fraction"] <= MAX_TREE_DEPTH_FRACTION
        and record["nonfinite_diagnostics"] == 0
        and record["max_rhat"] < gate["rhat_max"]
        and record["min_ess_bulk"] >= gate["ess_min"] and record["min_ess_tail"] >= gate["ess_min"])
    return record, families

idata = None
if STAGE == "fit":
    predecessor = PREDECESSOR[PROFILE]
    if predecessor is not None and SAMPLER_OVERRIDE is None:
        path = gate_path(predecessor)
        passed = path.exists() and json.loads(path.read_text()).get("gate_pass")
        if not passed:
            if not PROMOTION_WAIVER:
                raise RuntimeError(f"{PROFILE} requires a passed {predecessor} gate ({path}); "
                                   "see PROMOTION_WAIVER in cell 3")
            smoke_file = gate_path("smoke")
            if not smoke_file.exists():
                raise RuntimeError("A waiver needs a completed smoke fit")
            smoke = json.loads(smoke_file.read_text())
            geometry_ok = (smoke["divergences"] == 0 and smoke["min_bfmi"] >= BFMI_MIN
                           and smoke["max_maxdepth_fraction"] <= MAX_TREE_DEPTH_FRACTION
                           and smoke["nonfinite_diagnostics"] == 0 and smoke["max_rhat"] < SMOKE_RHAT_HARD_MAX
                           and smoke["code_test_override"] is None)
            if not geometry_ok:
                raise RuntimeError("Waiver refused: the smoke fit has a sampler-geometry problem")
            write_json({"profile": PROFILE, "failed_or_missing_predecessor": predecessor,
                        "predecessor_record": json.loads(path.read_text()) if path.exists() else None,
                        "smoke_record": smoke, "reason": PROMOTION_WAIVER,
                        "binding_gate": "publication: R-hat < 1.01, bulk/tail ESS >= 400, 0 divergences"},
                       OUTPUT_DIR / "promotion_waiver.json")
            print("Promotion waiver recorded:", PROMOTION_WAIVER)
    if TRACE_PATH.exists():
        raise FileExistsError(f"{TRACE_PATH} exists; never overwrite a trace")
    sampler = dict(PROFILES[PROFILE])
    if SAMPLER_OVERRIDE:
        sampler.update(SAMPLER_OVERRIDE)
        warnings.warn("SAMPLER_OVERRIDE is set: this run is a code test and can never pass a gate")
    start = time.time()
    with model:
        idata = pm.sample(tune=sampler["tune"], draws=sampler["draws"], chains=sampler["chains"],
                          cores=sampler["chains"], target_accept=sampler["target_accept"],
                          nuts_sampler="nutpie", nuts_sampler_kwargs={"maxdepth": MAX_TREE_DEPTH},
                          random_seed=RUN_SEED, return_inferencedata=True, progressbar=False)
    elapsed = time.time() - start
    idata.attrs.update({"run_name": RUN_NAME, "profile": PROFILE, "notebook_version": NOTEBOOK_VERSION})
    sanitize_attrs(idata)
    tmp = TRACE_PATH.with_name(".tmp_" + TRACE_PATH.name)
    idata.to_netcdf(tmp)
    tmp.replace(TRACE_PATH)
    print(f"Sampling took {elapsed / 60:.1f} min")
elif STAGE in {"diagnostics", "ppc", "compare"}:
    trace_profile = "publication" if STAGE == "compare" else PROFILE
    path = RUN_ROOT / trace_profile / "posterior_trace.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    idata = az.from_netcdf(path)

if STAGE in {"fit", "diagnostics"}:
    diagnostics, family_table = compute_diagnostics(idata, PROFILE)
    write_json(diagnostics, OUTPUT_DIR / "run_diagnostics.json")
    write_csv(family_table, OUTPUT_DIR / "diagnostic_families.csv")
    print(json.dumps(diagnostics, indent=2))
    print(family_table.head(10).to_string(index=False))

# 7. Draw-matched posterior predictive check (incomplete grid)
def nan_residual_stats(matrix):
    """matrix: draws x features x patients with NaN for removed cells."""
    sd = np.nanstd(matrix, axis=2, ddof=1)
    centred = matrix - np.nanmean(matrix, axis=2, keepdims=True)
    tail = np.nanmax(np.abs(centred), axis=2) / sd
    return sd, tail

def dependence_count(matrix, columns):
    x = matrix[:, columns, :]
    if x.shape[1] < 2:
        return np.zeros(x.shape[0], dtype=np.int32)
    x = x - np.nanmean(x, axis=2, keepdims=True)
    x = np.nan_to_num(x, nan=0.0)
    norms = np.sqrt((x * x).sum(axis=2))
    corr = np.einsum("bip,bjp->bij", x, x, optimize=True) / (norms[:, :, None] * norms[:, None, :])
    upper = np.triu_indices(x.shape[1], k=1)
    return (np.abs(corr[:, upper[0], upper[1]]) >= PPC_CORRELATION_THRESHOLD).sum(axis=1).astype(np.int32)

def upper_p(reference, observed):
    return float((1 + np.sum(reference >= observed)) / (reference.size + 1))

def lower_p(reference, observed):
    return float((1 + np.sum(reference <= observed)) / (reference.size + 1))

if STAGE == "ppc":
    diag_file = OUTPUT_DIR / "run_diagnostics.json"
    if not diag_file.exists():
        raise RuntimeError("Run the diagnostics for this profile first")
    diag_record = json.loads(diag_file.read_text())
    if not diag_record["gate_pass"] and SAMPLER_OVERRIDE is None:
        raise RuntimeError("PPC requires a trace that passed its convergence gate")

    data = arrays["data"]
    features, patients = arrays["features"], arrays["patients"]
    n_f, n_p, n_c = len(features), len(patients), len(arrays["celltypes"])
    f_idx = data["feature_idx"].to_numpy(int)
    p_idx = data["patient_idx"].to_numpy(int)
    c_idx = data["celltype_idx"].to_numpy(int)
    y_obs = data["score_model"].to_numpy(float)
    bmi = data["bmi_standardized"].to_numpy(float)
    covariates = data[["age_z", "sex_centered"]].to_numpy(float)
    grid = np.full((n_f, n_p), -1, dtype=np.int64)
    grid[f_idx, p_idx] = np.arange(len(data))
    observed_mask = grid >= 0
    columns_by_celltype = [np.flatnonzero(arrays["feature_to_celltype"] == j) for j in range(n_c)]

    def to_grid(values):
        out = np.full((values.shape[0], n_f, n_p), np.nan)
        out[:, observed_mask] = values[:, grid[observed_mask]]
        return out

    post = idata.posterior
    n_chains, n_draws = post.sizes["chain"], post.sizes["draw"]
    requested = PPC_DRAWS_OVERRIDE or PPC_DRAWS[PROFILE]
    requested = min(requested - requested % (2 * n_chains), 2 * n_chains * (n_draws // 2))
    per_chain = requested // n_chains
    positions = np.linspace(0, n_draws - 1, per_chain).round().astype(int)
    chain_of = np.repeat(np.arange(n_chains), per_chain)
    draw_of = np.tile(positions, n_chains)
    rng = np.random.default_rng(RUN_SEED + 70_000)

    raw_obs_grid = to_grid(y_obs[None, :])
    observed_raw_dep = np.array([dependence_count(raw_obs_grid, c)[0] for c in columns_by_celltype])

    stats = {k: [] for k in ("rep_sd", "rep_tail", "obs_sd", "obs_tail")}
    dep = {k: [] for k in ("rep_raw", "rep_res", "obs_res")}
    for start in range(0, requested, PPC_BATCH):
        sel = slice(start, min(start + PPC_BATCH, requested))
        import xarray as xr
        pick = post.isel(chain=xr.DataArray(chain_of[sel], dims="k"), draw=xr.DataArray(draw_of[sel], dims="k"))
        g = lambda name, dims: np.asarray(pick[name].transpose("k", *dims), float)
        base = g("feature_baseline", ("feature",))
        slope = g("feature_bmi_slope", ("feature",))
        cslope = g("feature_covariate_slope", ("covariate", "feature"))
        w = g("celltype_intercept_w", ("patient", "celltype"))
        ctb = g("celltype_bmi_slope", ("celltype",))
        ctc = g("celltype_covariate_slope", ("covariate", "celltype"))
        pcov = arrays["patient_covariates"]
        v = w - (pcov[None, :, 0, None] * ctb[:, None, :]
                 + np.einsum("pc,bck->bpk", pcov[:, 1:], ctc, optimize=True))
        sigma = g("sigma_feature", ("feature",))
        nu = g("nu", ("celltype",)) if SPEC["nu_scope"] == "celltype" else g("nu", ())[:, None]
        mu = (base[:, f_idx] + slope[:, f_idx] * bmi + v[:, p_idx, c_idx]
              + np.einsum("oc,bco->bo", covariates, cslope[:, :, f_idx], optimize=True))
        nu_o = nu[:, c_idx] if SPEC["nu_scope"] == "celltype" else np.broadcast_to(nu, mu.shape)
        y_rep = mu + sigma[:, f_idx] * rng.standard_t(nu_o)

        rep_raw_grid = to_grid(y_rep)
        rep_res_grid = to_grid(y_rep - mu)
        obs_res_grid = to_grid(y_obs[None, :] - mu)
        s1, t1 = nan_residual_stats(rep_res_grid)
        s2, t2 = nan_residual_stats(obs_res_grid)
        stats["rep_sd"].append(s1); stats["rep_tail"].append(t1)
        stats["obs_sd"].append(s2); stats["obs_tail"].append(t2)
        dep["rep_raw"].append(np.column_stack([dependence_count(rep_raw_grid, c) for c in columns_by_celltype]))
        dep["rep_res"].append(np.column_stack([dependence_count(rep_res_grid, c) for c in columns_by_celltype]))
        dep["obs_res"].append(np.column_stack([dependence_count(obs_res_grid, c) for c in columns_by_celltype]))
        del rep_raw_grid, rep_res_grid, obs_res_grid, y_rep, mu
        gc.collect()
    stats = {k: np.concatenate(v) for k, v in stats.items()}
    dep = {k: np.concatenate(v) for k, v in dep.items()}

    calibration = np.zeros(requested, bool)
    for chain in range(n_chains):
        idx = np.flatnonzero(chain_of == chain)
        calibration[idx[::2]] = True
    evaluation = ~calibration

    labels = ["__GLOBAL__", *arrays["celltypes"]]
    rows = []

    def add(domain, obs_ct, rep_ct):
        obs_all = np.column_stack([obs_ct.sum(axis=1), obs_ct])
        rep_all = np.column_stack([rep_ct.sum(axis=1), rep_ct])
        for j, label in enumerate(labels):
            o, r = obs_all[:, j], rep_all[:, j]
            up = (1 + np.sum(r >= o)) / (len(r) + 1)
            lo = (1 + np.sum(r <= o)) / (len(r) + 1)
            rows.append({"domain": domain, "celltype": label,
                         "observed_median": float(np.median(o)), "replicated_median": float(np.median(r)),
                         "replicated_q025": float(np.quantile(r, 0.025)),
                         "replicated_q975": float(np.quantile(r, 0.975)),
                         "upper_p": float(up), "lower_p": float(lo),
                         "flag_upper": bool(up <= PPC_ALPHA)})

    def ct_counts(flags):
        return np.column_stack([flags[:, c].sum(axis=1) for c in columns_by_celltype])

    for domain, obs_key, rep_key in (("residual_dispersion", "obs_sd", "rep_sd"),
                                     ("studentized_residual_tails", "obs_tail", "rep_tail")):
        low, high = np.quantile(stats[rep_key][calibration], [0.025, 0.975], axis=0)
        obs_flags = (stats[obs_key][evaluation] < low) | (stats[obs_key][evaluation] > high)
        rep_flags = (stats[rep_key][evaluation] < low) | (stats[rep_key][evaluation] > high)
        add(domain, ct_counts(obs_flags), ct_counts(rep_flags))
        if domain == "studentized_residual_tails":
            obs_tail_rate = obs_flags.mean(axis=0)
    add("raw_outcome_dependence",
        np.broadcast_to(observed_raw_dep, (evaluation.sum(), n_c)), dep["rep_raw"][evaluation])
    add("conditional_residual_dependence", dep["obs_res"][evaluation], dep["rep_res"][evaluation])

    checks = pd.DataFrame(rows)
    feature_checks = pd.DataFrame({
        "global_feature": features,
        "global_celltype": np.asarray(arrays["celltypes"])[arrays["feature_to_celltype"]],
        "n_patients_fitted": observed_mask.sum(axis=1),
        "observed_tail_median": np.median(stats["obs_tail"], axis=0),
        "replicated_tail_q975": np.quantile(stats["rep_tail"], 0.975, axis=0),
        "tail_outside_rate": obs_tail_rate,
    })
    glob_tail = checks[(checks.domain == "studentized_residual_tails") & (checks.celltype == "__GLOBAL__")].iloc[0]
    summary = {
        "ppc_version": "draw-matched-incomplete-grid-v1", "family": FAMILY, "profile": PROFILE,
        "joint_draws": int(requested), "evaluation_draws": int(evaluation.sum()),
        "trace_sha256": sha256_file(TRACE_PATH),
        "formal": PROFILE != "smoke" and SAMPLER_OVERRIDE is None,
        "tail_upper_p": float(glob_tail.upper_p),
        "tail_adequacy_pass": bool(glob_tail.upper_p > PPC_ALPHA),
        "global_domains_flagged": checks.loc[(checks.celltype == "__GLOBAL__") & checks.flag_upper, "domain"].tolist(),
        "dispersion_note": ("residual_dispersion uses feature-specific sigma fitted to the same data, "
                            "so it is weak by construction; read lower_p as well as upper_p"),
        "dependence_note": ("correlations centre each feature on its fitted patients and zero-fill removed "
                            "cells; the same rule is applied to observed and replicated data"),
    }
    write_csv(checks, OUTPUT_DIR / "ppc_domain_checks.csv")
    write_csv(feature_checks, OUTPUT_DIR / "ppc_feature_checks.csv")
    write_json(summary, OUTPUT_DIR / "ppc_summary.json")
    print(checks[checks.celltype == "__GLOBAL__"].to_string(index=False))
    print(json.dumps(summary, indent=2))

def summarise_slopes(post, name, dim):
    da = post[name].transpose(dim, "chain", "draw")
    ids = np.asarray(da.coords[dim], str)
    s = np.asarray(da).reshape(len(ids), -1)
    h = az.hdi(post[name], hdi_prob=0.95)[name].sel({dim: ids}).values
    return pd.DataFrame({dim: ids, "mean": s.mean(axis=1), "hdi_low": h[:, 0], "hdi_high": h[:, 1],
                         "p_positive": (s > 0).mean(axis=1)})

if STAGE == "compare":
    pub_diag = RUN_ROOT / "publication" / "run_diagnostics.json"
    pub_ppc = RUN_ROOT / "publication" / "ppc_summary.json"
    for path in (pub_diag, pub_ppc):
        if not path.exists():
            raise FileNotFoundError(f"compare needs {path}")
    if not json.loads(pub_diag.read_text())["gate_pass"] and SAMPLER_OVERRIDE is None:
        raise RuntimeError("The ri_hc publication fit did not pass its convergence gate")
    primary_root = PRIMARY_LOWREAD_ROOT / PRIMARY_RUN_NAME / "publication"
    prim_trace = primary_root / "posterior_trace.nc"
    if not prim_trace.exists():
        raise FileNotFoundError(f"Published low-read primary trace not found: {prim_trace}")
    prim = az.from_netcdf(prim_trace)
    bmi_sd = float(cohort["bmi"].std(ddof=1)); per5 = 5.0 / bmi_sd

    labels = full_frame[["global_feature", "CellType", "Signature"]].drop_duplicates("global_feature")
    new = summarise_slopes(idata.posterior, "feature_bmi_slope", "feature").add_prefix("new_")
    old = summarise_slopes(prim.posterior, "feature_bmi_slope", "feature").add_prefix("prim_")
    merged = (labels.merge(old.rename(columns={"prim_feature": "global_feature"}), on="global_feature")
                    .merge(new.rename(columns={"new_feature": "global_feature"}), on="global_feature"))
    if len(merged) != SPEC["expected_features"]:
        raise ValueError("Feature sets do not align with the primary fit")
    for c in [c for c in merged.columns if c.endswith(("mean", "hdi_low", "hdi_high"))]:
        merged[c] = merged[c] * per5
    merged["prim_wide_low"] = merged.prim_mean + WIDENING * (merged.prim_hdi_low - merged.prim_mean)
    merged["prim_wide_high"] = merged.prim_mean + WIDENING * (merged.prim_hdi_high - merged.prim_mean)
    excl = lambda lo, hi: (lo > 0) | (hi < 0)
    merged["prim_excl"] = excl(merged.prim_hdi_low, merged.prim_hdi_high)
    merged["prim_wide_excl"] = excl(merged.prim_wide_low, merged.prim_wide_high)
    merged["new_excl"] = excl(merged.new_hdi_low, merged.new_hdi_high)
    merged["width_ratio"] = (merged.new_hdi_high - merged.new_hdi_low) / (merged.prim_hdi_high - merged.prim_hdi_low)
    merged["sign_flip"] = np.sign(merged.new_mean) != np.sign(merged.prim_mean)

    ct_new = summarise_slopes(idata.posterior, "celltype_bmi_slope", "celltype").add_prefix("new_")
    ct_old = summarise_slopes(prim.posterior, "celltype_bmi_slope", "celltype").add_prefix("prim_")
    celltype = ct_old.rename(columns={"prim_celltype": "celltype"}).merge(
        ct_new.rename(columns={"new_celltype": "celltype"}), on="celltype")
    for c in [c for c in celltype.columns if c.endswith(("mean", "hdi_low", "hdi_high"))]:
        celltype[c] = celltype[c] * per5
    celltype["prim_excl"] = excl(celltype.prim_hdi_low, celltype.prim_hdi_high)
    celltype["new_excl"] = excl(celltype.new_hdi_low, celltype.new_hdi_high)
    celltype["width_ratio"] = (celltype.new_hdi_high - celltype.new_hdi_low) / (celltype.prim_hdi_high - celltype.prim_hdi_low)
    sig = idata.posterior["celltype_intercept_sigma"].mean(("chain", "draw")).to_pandas()
    celltype["intercept_sigma_mean"] = celltype["celltype"].map(sig)

    dep_cols = ["celltype", "observed_median", "replicated_q975", "upper_p", "flag_upper"]
    dep_new = pd.read_csv(RUN_ROOT / "publication" / "ppc_domain_checks.csv")
    dep_old_path = primary_root / "ppc_domain_checks.csv"
    dep_new = dep_new[dep_new.domain.eq("conditional_residual_dependence")][dep_cols]
    if dep_old_path.exists():
        dep_old = pd.read_csv(dep_old_path)
        dep_old = dep_old[dep_old.domain.eq("conditional_residual_dependence")][dep_cols]
        dependence = dep_old.merge(dep_new, on="celltype", suffixes=("_prim", "_new"))
    else:
        dependence = dep_new.add_suffix("_new").rename(columns={"celltype_new": "celltype"})

    wanted = pd.DataFrame(HEADLINE[FAMILY], columns=["CellType", "Signature"])
    headline = wanted.merge(merged, on=["CellType", "Signature"], how="left")
    missing = headline[headline.global_feature.isna()][["CellType", "Signature"]].values.tolist()

    agreement = pd.crosstab(merged.prim_wide_excl, merged.new_excl,
                            rownames=["primary_1.37x_excludes_0"], colnames=["ri_hc_excludes_0"])
    glob_flag = lambda d, col: bool(d.loc[d.celltype.eq("__GLOBAL__"), col].iloc[0]) if col in d else None
    dep_resolved = (glob_flag(dependence, "flag_upper_new") is False)
    decision = {
        "family": FAMILY, "model": "ri_replace_hc", "max_tree_depth": MAX_TREE_DEPTH,
        "units": "score SD per 5 kg/m2",
        "n_hdi_excluding_zero": {"primary": int(merged.prim_excl.sum()),
                                 "primary_widened_1.37": int(merged.prim_wide_excl.sum()),
                                 "ri_hc": int(merged.new_excl.sum()),
                                 "expected_if_all_null": round(0.05 * len(merged), 1)},
        "agreement_widened_vs_ri_hc": agreement.to_dict(),
        "sign_flips_among_excluding": int((merged.sign_flip & (merged.prim_excl | merged.new_excl)).sum()),
        "median_feature_width_ratio": float(merged.width_ratio.median()),
        "celltype_means_excluding_zero": {"primary": int(celltype.prim_excl.sum()), "ri_hc": int(celltype.new_excl.sum())},
        "dependence_global_flag": {"primary": glob_flag(dependence, "flag_upper_prim"),
                                   "ri_hc": glob_flag(dependence, "flag_upper_new")},
        "dependence_celltypes_flagged": {
            "primary": int(dependence.loc[dependence.celltype.ne("__GLOBAL__"), "flag_upper_prim"].sum()) if "flag_upper_prim" in dependence else None,
            "ri_hc": int(dependence.loc[dependence.celltype.ne("__GLOBAL__"), "flag_upper_new"].sum())},
        "headline_missing": missing,
        "ppc_summary": json.loads(pub_ppc.read_text()),
        "primary_trace_sha256": sha256_file(prim_trace),
        "ri_hc_trace_sha256": sha256_file(RUN_ROOT / "publication" / "posterior_trace.nc"),
    }
    decision["triggered"] = {"dependence_still_flagged": not dep_resolved}
    decision["decision"] = ("DEPENDENCE_RESOLVED" if dep_resolved else "DEPENDENCE_STILL_FLAGGED")
    compare_dir = RUN_ROOT / "comparison"
    compare_dir.mkdir(exist_ok=True)
    write_csv(merged, compare_dir / "feature_comparison.csv")
    write_csv(celltype, compare_dir / "celltype_comparison.csv")
    write_csv(dependence, compare_dir / "dependence_comparison.csv")
    write_csv(headline, compare_dir / "headline_comparison.csv")
    write_json(decision, compare_dir / "decision.json")
    show = ["CellType", "Signature", "prim_mean", "prim_hdi_low", "prim_hdi_high", "prim_wide_low",
            "new_mean", "new_hdi_low", "new_hdi_high", "width_ratio"]
    print(json.dumps({k: decision[k] for k in ("decision", "n_hdi_excluding_zero", "celltype_means_excluding_zero",
                                                "dependence_global_flag", "dependence_celltypes_flagged",
                                                "median_feature_width_ratio")}, indent=2))
    print(celltype.round(3).to_string(index=False))
    print(dependence.round(3).to_string(index=False))
    print(headline[show].round(3).to_string(index=False))


In [ ]:
%%writefile /content/k1_worker.py
import json, time, gc, hashlib, re
import numpy as np, pandas as pd, arviz as az
import pymc as pm, pytensor.tensor as pt
from pathlib import Path
from scipy import stats

# ---------------- settings ----------------
import os
N_SHARDS = 5
SHARD_ID = int(os.environ.get("K1_SHARD_ID", "0"))
N_WORKERS = int(os.environ.get("K1_N_WORKERS", "1"))
WORKER_ID = int(os.environ.get("K1_WORKER_ID", "0"))
RUN_SIMULATION = os.environ.get("K1_RUN", "1") == "1"
MAX_NEW_MCMC = None
assert 0 <= SHARD_ID < N_SHARDS and 0 <= WORKER_ID < N_WORKERS
GEOMETRY = "fine4"
REPLICATES = 20
SCENARIOS = ["null_corr", "fitted_corr", "sparse_corr", "positive_copula", "contamination", "small_n", "matched_factor"]
MODELS = ["hc_t", "factor_K1"]
SCENARIO_MODELS = {"matched_factor": ["factor_K1"]}
PATIENT_NU = 4.0
COPULA_RHO = 0.6
SPARSE_SHIFT, SPARSE_FRACTION, SPARSE_DEV_SD = 0.12, 0.15, 0.08
CONTAM_PROFILE_FRACTION, CONTAM_PROFILE_SHIFT = 0.05, 3.0
CONTAM_OBS_FRACTION, CONTAM_OBS_SHIFT = 0.01, 5.0
SMALL_N = 80
LOADING_SD = 0.30
SAMPLER = dict(tune=2000, draws=2000, chains=4, target_accept=0.99, maxdepth=10)
RETRY_SAMPLER = dict(tune=4000, draws=3000, chains=4, target_accept=0.995, maxdepth=10)
GATE = dict(rhat=1.01, ess=400, bfmi=0.3, maxdepth_fraction=0.01)
SEED = 20261118
GEOMETRIES = {"fine4": ("immune_fine", ["TAM", "CD8_T_GENERAL", "B_CELLS", "NK_CELLS"]),
              "coarse4": ("immune_coarse", ["CD4+ T CELLS", "CD8_T_GENERAL", "MONOCYTES", "B_CELLS"])}
BASE = Path("/content/drive/MyDrive/new_dysfunction")
STUDY = f"k1_validation_{GEOMETRY}_mount5_cpu_v1"
SIM_ROOT = BASE / STUDY; (SIM_ROOT / "tasks").mkdir(parents=True, exist_ok=True)

CONFIG = dict(study=STUDY, geometry=GEOMETRY, replicates=REPLICATES, scenarios=SCENARIOS, models=MODELS,
              scenario_models=SCENARIO_MODELS, patient_nu=PATIENT_NU, copula_rho=COPULA_RHO,
              sparse=[SPARSE_SHIFT, SPARSE_FRACTION, SPARSE_DEV_SD],
              contamination=[CONTAM_PROFILE_FRACTION, CONTAM_PROFILE_SHIFT, CONTAM_OBS_FRACTION, CONTAM_OBS_SHIFT],
              small_n=SMALL_N, loading_sd=LOADING_SD, sampler=SAMPLER, retry_sampler=RETRY_SAMPLER, gate=GATE, seed=SEED, n_shards=N_SHARDS, sampler_backend="numba")
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]
cfg_path = SIM_ROOT / "study_config.json"
if cfg_path.exists():
    assert json.loads(cfg_path.read_text())["config_hash"] == CONFIG_HASH, \
        "This folder holds a study with DIFFERENT settings. Change STUDY or restore the settings."
else:
    cfg_path.write_text(json.dumps({**CONFIG, "config_hash": CONFIG_HASH}, indent=2))

FAMILY_NAME, SIM_CELLTYPES = GEOMETRIES[GEOMETRY]
src = Path("/content/ri_hc_engine.py").read_text()
src, n_sub = re.subn(r'RUN_NAME = f"\{PRIMARY_RUN_NAME\}__ri_hc[^"]*"',
                     f'RUN_NAME = f"{{PRIMARY_RUN_NAME}}__k1_validation_mount5_shard{SHARD_ID}_w{WORKER_ID}"', src)
assert n_sub == 1, "could not redirect the engine's run name"
ns = {"CONFIG": {"family": FAMILY_NAME, "profile": "smoke", "stage": "preflight", "waiver": None,
                 "max_tree_depth": 10}, "__name__": "lowread_engine"}
exec(compile(src, "lowread_engine.py", "exec"), ns)
SPEC = ns["SPEC"]; full = ns["arrays"]
keep = [f"{SPEC['compartment']}||{c}" for c in SIM_CELLTYPES]
assert set(keep) <= set(full["celltypes"]), set(keep) - set(full["celltypes"])
arrays = ns["prepare_arrays"](full["data"][full["data"]["global_celltype"].isin(keep)].copy())
data = arrays["data"]
F_IDX, P_IDX, C_IDX = (data[c].to_numpy(int) for c in ("feature_idx", "patient_idx", "celltype_idx"))
f2c = arrays["feature_to_celltype"]
feats, cts, pats = list(arrays["features"]), list(arrays["celltypes"]), list(arrays["patients"])
n_f, n_p, n_c = len(feats), len(pats), len(cts)
pcov = arrays["patient_covariates"]
BMI_OBS = data["bmi_standardized"].to_numpy(float)
COV_OBS = data[["age_z", "sex_centered"]].to_numpy(float)
NU_SCOPE = SPEC["nu_scope"]
PER5 = 5.0 / float(ns["cohort"]["bmi"].std(ddof=1))
print(f"geometry {GEOMETRY}: {n_f} features, {n_c} cell types, {len(data):,} observations, nu scope {NU_SCOPE}")

truth_path = ns["SENS_ROOT"] / f"{ns['PRIMARY_RUN_NAME']}__ri_hc_t" / "publication" / "posterior_trace.nc"
post = az.from_netcdf(truth_path).posterior
pmean = lambda n: post[n].mean(("chain", "draw"))
T = {"base": pmean("feature_baseline").sel(feature=feats).values,
     "cov_slope": pmean("feature_covariate_slope").sel(feature=feats).transpose("covariate", "feature").values,
     "sigma": pmean("sigma_feature").sel(feature=feats).values,
     "ct_slope": pmean("celltype_bmi_slope").sel(celltype=cts).values,
     "dev_sd": pmean("feature_sigma_bmi").sel(celltype=cts).values,
     "int_sd": pmean("celltype_intercept_sigma").sel(celltype=cts).values,
     "int_nu": float(pmean("intercept_nu").values)}
nu_post = pmean("nu")
T["nu_c"] = (nu_post.sel(celltype=cts).values if "celltype" in nu_post.dims else np.full(n_c, float(nu_post.values)))
w_mean = pmean("celltype_intercept_w").sel(patient=pats, celltype=cts).values
ctc = pmean("celltype_covariate_slope").sel(celltype=cts).transpose("covariate", "celltype").values
v_mean = w_mean - (pcov[:, [0]] * T["ct_slope"][None, :] + pcov[:, 1:] @ ctc)
mu_fit = (T["base"][F_IDX] + pmean("feature_bmi_slope").sel(feature=feats).values[F_IDX] * BMI_OBS
          + v_mean[P_IDX, C_IDX] + np.einsum("oc,co->o", COV_OBS, T["cov_slope"][:, F_IDX]))
resid = pd.DataFrame({"f": F_IDX, "p": P_IDX, "r": data["score_model"].to_numpy() - mu_fit}).pivot(index="p", columns="f", values="r")
del post; gc.collect()

CHOL, LOAD, EQUI = [], [], []
for j in range(n_c):
    cols = np.flatnonzero(f2c == j)
    R = resid[cols].corr(min_periods=30).fillna(0.0).to_numpy(); np.fill_diagonal(R, 1.0)
    val, vec = np.linalg.eigh(R); val = np.clip(val, 1e-3, None)
    R = (vec * val) @ vec.T; d = np.sqrt(np.diag(R)); R = R / np.outer(d, d)
    CHOL.append((cols, np.linalg.cholesky(R)))
    LOAD.append((cols, vec[:, -1] * np.sqrt(val[-1])))
    E = np.full((len(cols), len(cols)), COPULA_RHO); np.fill_diagonal(E, 1.0)
    EQUI.append((cols, np.linalg.cholesky(E)))
print("truth: intercept nu %.2f; obs nu %s" % (T["int_nu"], np.round(T["nu_c"], 2)))

# ---------------- simulation ----------------
def simulate(scenario, rep):
    rng = np.random.default_rng([SEED, SCENARIOS.index(scenario), rep])
    nu_f = T["nu_c"][f2c]
    if scenario == "null_corr":
        m_c, dev = np.zeros(n_c), np.zeros(n_f)
    elif scenario == "sparse_corr":
        m_c = np.zeros(n_c); m_c[rep % n_c] = SPARSE_SHIFT
        dev = np.where(rng.random(n_f) < SPARSE_FRACTION, rng.normal(0, SPARSE_DEV_SD, n_f), 0.0)
    else:
        m_c, dev = T["ct_slope"].copy(), rng.normal(0.0, T["dev_sd"][f2c])
    beta = m_c[f2c] + dev
    e = np.zeros((n_p, n_f))
    if scenario == "matched_factor":
        shared = (T["int_sd"][None, :] * rng.standard_t(T["int_nu"], size=(n_p, n_c)))[:, f2c]
        for cols, lam in LOAD:
            uniq = np.clip(1.0 - lam ** 2, 0.05, None)
            fac = rng.standard_normal((n_p, 1)) * lam[None, :]
            e[:, cols] = T["sigma"][cols] * (fac + np.sqrt(uniq) * rng.standard_t(nu_f[cols], size=(n_p, len(cols))))
    else:
        shared = (T["int_sd"][None, :] * rng.standard_t(PATIENT_NU, size=(n_p, n_c)))[:, f2c]
        blocks = EQUI if scenario == "positive_copula" else CHOL
        for j, (cols, L) in enumerate(blocks):
            z = rng.standard_normal((n_p, len(cols))) @ L.T
            scale = np.sqrt(T["nu_c"][j] / rng.chisquare(T["nu_c"][j], size=(n_p, 1)))
            e[:, cols] = z * scale * T["sigma"][cols]
    grid = T["base"][None, :] + pcov[:, [0]] * beta[None, :] + pcov[:, 1:] @ T["cov_slope"] + shared + e
    if scenario == "contamination":
        bad = rng.random((n_p, n_c)) < CONTAM_PROFILE_FRACTION
        sign = rng.choice([-1.0, 1.0], size=(n_p, n_c))
        grid += (bad * sign * CONTAM_PROFILE_SHIFT * T["int_sd"][None, :])[:, f2c]
        badobs = rng.random((n_p, n_f)) < CONTAM_OBS_FRACTION
        grid += badobs * rng.choice([-1.0, 1.0], size=(n_p, n_f)) * CONTAM_OBS_SHIFT * T["sigma"][None, :]
    obs = np.arange(len(F_IDX))
    if scenario == "small_n":
        kept = np.sort(rng.choice(n_p, size=SMALL_N, replace=False))
        obs = obs[np.isin(P_IDX, kept)]
    return grid[P_IDX[obs], F_IDX[obs]], obs, beta, m_c

def build(n_factors, y, obs):
    P = ns
    fidx, pidx, cidx = F_IDX[obs], P_IDX[obs], C_IDX[obs]
    coords = {"obs_id": np.arange(len(y)), "feature": feats, "celltype": cts, "patient": pats,
              "covariate": ["age_z", "sex_centered"]}
    with pm.Model(coords=coords) as model:
        fi, pi, ci = (pm.Data(n, a, dims="obs_id") for n, a in (("feature_idx", fidx), ("patient_idx", pidx), ("celltype_idx", cidx)))
        bmi = pm.Data("bmi_z", BMI_OBS[obs], dims="obs_id")
        cov = pm.Data("cov", COV_OBS[obs], dims=("obs_id", "covariate"))
        base = pm.Normal("feature_baseline", 0.0, P["BASELINE_SD"], dims="feature")
        ct = pm.Normal("celltype_bmi_slope", 0.0, P["CELLTYPE_BMI_SD"], dims="celltype")
        fs = pm.HalfNormal("feature_sigma_bmi", P["FEATURE_BMI_SD"], dims="celltype")
        raw = pm.Normal("feature_bmi_raw", 0.0, 1.0, dims="feature")
        dev = raw * fs[f2c]
        pm.Deterministic("feature_bmi_slope", ct[f2c] + dev, dims="feature")
        ctc_ = pm.Normal("celltype_covariate_slope", 0.0, P["CELLTYPE_COVARIATE_SD"], dims=("covariate", "celltype"))
        fsc = pm.HalfNormal("feature_sigma_covariate", P["FEATURE_COVARIATE_SD"], dims=("covariate", "celltype"))
        rawc = pm.Normal("feature_covariate_raw", 0.0, 1.0, dims=("covariate", "feature"))
        pb = pm.Data("patient_bmi", pcov[:, 0], dims="patient")
        pc_ = pm.Data("patient_cov", pcov[:, 1:], dims=("patient", "covariate"))
        isd = pm.HalfNormal("celltype_intercept_sigma", P["CELLTYPE_INTERCEPT_SD"], dims="celltype")
        inu = pm.Deterministic("intercept_nu", pm.Exponential("intercept_nu_minus_two", lam=P["NU_MINUS_TWO_RATE"]) + 2.0)
        wm = pb[:, None] * ct[None, :] + pt.dot(pc_, ctc_)
        w = pm.StudentT("celltype_intercept_w", nu=inu, mu=wm, sigma=isd[None, :], dims=("patient", "celltype"))
        mu = base[fi] + dev[fi] * bmi + w[pi, ci] + pt.sum(cov * (rawc * fsc[:, f2c])[:, fi].T, axis=1)
        if n_factors:
            lsd = pm.HalfNormal("loading_sd", LOADING_SD, dims="celltype")
            lraw = pm.Normal("loading_raw", 0.0, 1.0, dims="feature")
            lam = lraw * lsd[f2c]
            pm.Deterministic("factor_variance", lam ** 2, dims="feature")
            fac = pm.Normal("factor_score", 0.0, 1.0, dims=("patient", "celltype"))
            mu = mu + lam[fi] * fac[pi, ci]
        lsl = pm.Normal("log_sigma_location", P["LOG_SIGMA_LOCATION"], P["LOG_SIGMA_LOCATION_SD"])
        lcs = pm.HalfNormal("log_sigma_celltype_location_sd", P["CELLTYPE_LOG_SIGMA_SD"])
        lcr = pm.Normal("log_sigma_celltype_raw", 0.0, 1.0, dims="celltype")
        lfs = pm.HalfNormal("log_sigma_feature_scale", P["FEATURE_LOG_SIGMA_SD"], dims="celltype")
        lr = pm.Normal("log_sigma_raw", 0.0, 1.0, dims="feature")
        sig = pm.Deterministic("sigma_feature", pt.exp((lsl + lcs * lcr)[f2c] + lfs[f2c] * lr), dims="feature")
        if NU_SCOPE == "celltype":
            nu = pm.Deterministic("nu", pm.Exponential("nu_minus_two", lam=P["NU_MINUS_TWO_RATE"], dims="celltype") + 2.0, dims="celltype")
            nu_o = nu[ci]
        else:
            nu = pm.Deterministic("nu", pm.Exponential("nu_minus_two", lam=P["NU_MINUS_TWO_RATE"]) + 2.0)
            nu_o = nu
        pm.StudentT("y_obs", nu=nu_o, mu=mu, sigma=sig[fi], observed=y, dims="obs_id")
    return model

IDENTIFIED = ["feature_bmi_slope", "celltype_bmi_slope", "sigma_feature", "celltype_intercept_sigma", "intercept_nu", "nu"]

def az_vals(func, da, **kw):
    return np.asarray(func(da.to_dataset(name="v"), **kw)["v"].values, float)

def interval_metrics(da, truth, dim):
    h = az_vals(az.hdi, da, hdi_prob=0.95)
    h = h.reshape(-1, 2) if h.ndim == 2 else h
    lo, hi = h[:, 0], h[:, 1]
    est = da.mean(("chain", "draw")).values
    call = (lo > 0) | (hi < 0); null = np.isclose(truth, 0.0)
    return {"coverage": float(((lo <= truth) & (truth <= hi)).mean()), "mean_width": float(np.mean(hi - lo)),
            "n_calls": int(call.sum()), "false_calls": int((call & null).sum()), "n_null": int(null.sum()),
            "true_calls": int((call & ~null & (np.sign(est) == np.sign(truth))).sum()), "n_nonnull": int((~null).sum()),
            "rmse": float(np.sqrt(np.mean((est - truth) ** 2))), "bias": float(np.mean(est - truth))}

def diagnostics(idata, n_factors):
    p = idata.posterior; st = idata.sample_stats
    names = IDENTIFIED + (["factor_variance"] if n_factors else [])
    r = max(float(np.nanmax(az_vals(az.rhat, p[v]))) for v in names)
    eb = min(float(np.nanmin(az_vals(az.ess, p[v], method="bulk"))) for v in names)
    et = min(float(np.nanmin(az_vals(az.ess, p[v], method="tail"))) for v in names)
    bfmi = float(np.min(az.bfmi(idata)))
    depth = float(st["maxdepth_reached"].values.mean() if "maxdepth_reached" in st
                  else (st["tree_depth"].values >= SAMPLER["maxdepth"]).mean())
    div = int(st["diverging"].values.sum())
    ok = r < GATE["rhat"] and eb >= GATE["ess"] and et >= GATE["ess"] and div == 0 and bfmi >= GATE["bfmi"] and depth <= GATE["maxdepth_fraction"]
    reason = ("divergences" if div else "rhat" if r >= GATE["rhat"] else "ess" if min(eb, et) < GATE["ess"]
              else "bfmi" if bfmi < GATE["bfmi"] else "treedepth" if depth > GATE["maxdepth_fraction"] else "")
    return dict(max_rhat=r, min_ess_bulk=eb, min_ess_tail=et, divergences=div, min_bfmi=bfmi,
                maxdepth_fraction=depth, converged=bool(ok), failure_reason=reason)

def composite_hc3(y, obs, beta):
    """Patient-level benchmark: per cell type, mean signature score per patient, OLS on BMI + age + sex, HC3."""
    frame = pd.DataFrame({"y": y, "p": P_IDX[obs], "c": C_IDX[obs]})
    out = []
    for j in range(n_c):
        g = frame[frame.c == j].groupby("p").y.mean()
        X = np.column_stack([np.ones(len(g)), pcov[g.index.to_numpy()]])
        XtXi = np.linalg.inv(X.T @ X); b = XtXi @ X.T @ g.to_numpy()
        res = g.to_numpy() - X @ b; h = np.einsum("ij,jk,ik->i", X, XtXi, X)
        V = XtXi @ (X.T * (res / (1 - h)) ** 2) @ X @ XtXi
        se = np.sqrt(V[1, 1]); crit = stats.t.ppf(0.975, len(g) - 4)
        out.append((b[1] - crit * se, b[1] + crit * se, b[1]))
    lo, hi, est = map(np.array, zip(*out))
    truth = np.array([beta[f2c == j].mean() for j in range(n_c)])
    call = (lo > 0) | (hi < 0); null = np.isclose(truth, 0.0)
    return {"coverage": float(((lo <= truth) & (truth <= hi)).mean()), "mean_width": float(np.mean(hi - lo)),
            "n_calls": int(call.sum()), "false_calls": int((call & null).sum()), "n_null": int(null.sum()),
            "true_calls": int((call & ~null & (np.sign(est) == np.sign(truth))).sum()), "n_nonnull": int((~null).sum()),
            "rmse": float(np.sqrt(np.mean((est - truth) ** 2))), "bias": float(np.mean(est - truth))}

def fit_once(n_factors, y, obs, sampler, seed):
    t0 = time.time()
    with build(n_factors, y, obs):
        idata = pm.sample(tune=sampler["tune"], draws=sampler["draws"], chains=sampler["chains"], cores=sampler["chains"],
                          target_accept=sampler["target_accept"], nuts_sampler="nutpie",
                          nuts_sampler_kwargs={"maxdepth": sampler["maxdepth"]}, random_seed=seed,
                          return_inferencedata=True, progressbar=False)
    return idata, (time.time() - t0) / 60

def task_key(scenario, model, rep):
    return f"{scenario}__{model}__r{rep:03d}"

def run_task(scenario, model, rep):
    out = SIM_ROOT / "tasks" / f"{task_key(scenario, model, rep)}.json"
    if out.exists():
        previous = json.loads(out.read_text())
        assert previous.get("config_hash") == CONFIG_HASH, f"Config mismatch in {out}"
        assert (previous.get("scenario"), previous.get("model"), previous.get("replicate")) == (scenario, model, rep), f"Task mismatch in {out}"
        return previous
    y, obs, beta, m_c = simulate(scenario, rep)
    rec = {"scenario": scenario, "model": model, "replicate": rep, "config_hash": CONFIG_HASH,
           "data_sha256": hashlib.sha256(np.concatenate([y, obs.astype(float)]).tobytes()).hexdigest(),
           "n_observations": int(len(y)), "per5": PER5}
    if model == "composite":
        rec.update({"attempts": [], "final_attempt": 0, "converged": True, "minutes": 0.0,
                    "celltype": composite_hc3(y, obs, beta)})
    else:
        k = 1 if model == "factor_K1" else 0
        attempts = []
        for attempt, sampler in enumerate([SAMPLER, RETRY_SAMPLER]):
            seed = SEED + 7919 * rep + 104729 * MODELS.index(model) + 13 * SCENARIOS.index(scenario) + 1_000_003 * attempt
            idata, minutes = fit_once(k, y, obs, sampler, seed)
            d = diagnostics(idata, k)
            p = idata.posterior
            attempts.append({"attempt": attempt, "sampler": sampler, "minutes": minutes, **d,
                             "feature": interval_metrics(p["feature_bmi_slope"], beta, "feature"),
                             "celltype": interval_metrics(p["celltype_bmi_slope"], m_c, "celltype")})
            del idata, p; gc.collect()
            if d["converged"]:
                break
        rec.update({"attempts": attempts, "final_attempt": len(attempts) - 1,
                    "converged_first": attempts[0]["converged"], "converged": attempts[-1]["converged"],
                    "minutes": sum(a["minutes"] for a in attempts)})
    tmp = out.with_suffix(".tmp"); tmp.write_text(json.dumps(rec, indent=2)); tmp.replace(out)
    return rec

TASKS = []
for r in range(REPLICATES):
    for s in SCENARIOS:
        TASKS.append((s, "composite", r))
        for m in SCENARIO_MODELS.get(s, MODELS):
            TASKS.append((s, m, r))
mcmc = [t for t in TASKS if t[1] != "composite"]
MINE = [t for i, t in enumerate(mcmc) if i % N_SHARDS == SHARD_ID]
if SHARD_ID == 0:
    MINE = MINE + [t for t in TASKS if t[1] == "composite"]
SHARD_TASKS = list(MINE)
if RUN_SIMULATION and N_WORKERS > 1:
    fits = [t for t in MINE if t[1] != "composite"]
    MINE = [t for i, t in enumerate(fits) if i % N_WORKERS == WORKER_ID]
    if WORKER_ID == 0:
        MINE += [t for t in SHARD_TASKS if t[1] == "composite"]
MINE = [t for t in MINE if t[1] != "factor_K1"]
print(f"study {STUDY} [{CONFIG_HASH}] shard {SHARD_ID}/{N_SHARDS} worker {WORKER_ID}/{N_WORKERS}: {len(MINE)} tasks "
      f"({sum(t[1] == 'factor_K1' for t in MINE)} K1, {sum(t[1] == 'hc_t' for t in MINE)} hc_t)")
if RUN_SIMULATION:
    new_mcmc = 0
    for s, m, r in MINE:
        if MAX_NEW_MCMC is not None and new_mcmc >= MAX_NEW_MCMC:
            break
        checkpoint = SIM_ROOT / "tasks" / f"{task_key(s, m, r)}.json"
        already_done = checkpoint.exists()
        rec = run_task(s, m, r)
        if m != "composite" and not already_done:
            new_mcmc += 1
        if m == "composite":
            continue
        a = rec["attempts"][rec["final_attempt"]]
        print(f"{s:16s} {m:10s} r{r:02d}  attempts={len(rec['attempts'])} conv={rec['converged']} "
              f"rhat={a['max_rhat']:.3f} div={a['divergences']}  feat cov={a['feature']['coverage']:.3f}  "
              f"ct cov={a['celltype']['coverage']:.3f}  ({rec['minutes']:.1f} min)", flush=True)
done = sum((SIM_ROOT / "tasks" / f"{task_key(*t)}.json").exists() for t in SHARD_TASKS)
print(f"shard {SHARD_ID}: {done}/{len(SHARD_TASKS)} tasks finished", flush=True)


In [ ]:
%%writefile /content/sim2_worker.py
import os, json, time, gc, hashlib
import numpy as np, pandas as pd, arviz as az, pymc as pm, pytensor.tensor as pt
from pathlib import Path
from scipy import stats
from scipy.stats import false_discovery_control

os.environ.update(K1_SHARD_ID="0", K1_N_WORKERS="1", K1_WORKER_ID="0", K1_RUN="0")
G = {"__name__": "k1_setup"}
exec(compile(Path("/content/k1_worker.py").read_text(), "k1_worker.py", "exec"), G)
for k in ["SIM_ROOT", "SEED", "SCENARIOS", "REPLICATES", "SAMPLER", "RETRY_SAMPLER", "GATE", "simulate", "interval_metrics",
          "fit_once", "az_vals", "F_IDX", "P_IDX", "C_IDX", "f2c", "feats", "cts", "pats", "n_f", "n_p", "n_c", "pcov",
          "BMI_OBS", "COV_OBS", "NU_SCOPE", "T", "CHOL", "PATIENT_NU", "data", "CONFIG_HASH"]:
    globals()[k] = G[k]

SHARD, NSHARD = int(os.environ.get("SIM2_SHARD", "0")), int(os.environ.get("SIM2_NSHARDS", "1"))
WORKER, NWORKER = int(os.environ.get("SIM2_WORKER", "0")), int(os.environ.get("SIM2_NWORKERS", "1"))
RUN = os.environ.get("SIM2_RUN", "1") == "1"
OUT = SIM_ROOT / "tasks_v2"; OUT.mkdir(exist_ok=True)
SC6 = [s for s in SCENARIOS if s != "matched_factor"]
GRID = [0.05, 0.10, 0.20, 0.30]
CAT_CT_SD, CAT_F_SD = 0.40, 0.50
def _patient_groups():
    ns = G.get("ns", {}) if isinstance(G.get("ns"), dict) else {}
    frames = [data]
    if isinstance(G.get("full"), dict) and "data" in G["full"]: frames.append(G["full"]["data"])
    frames += [ns.get(k) for k in ("cohort", "full_frame", "covariates") if ns.get(k) is not None]
    for f in frames:
        if "base_sample_id" not in f: continue
        if "bmi_group" in f:
            m = f.drop_duplicates("base_sample_id").set_index("base_sample_id")["bmi_group"]
        elif "bmi" in f:
            b = f.drop_duplicates("base_sample_id").set_index("base_sample_id")["bmi"]
            m = pd.Series(np.where(b >= 30, "obese", np.where(b >= 25, "overweight", "normal_weight")), index=b.index)
        else:
            continue
        g = m.reindex(pats).to_numpy()
        if pd.notna(g).all(): return g
    raise RuntimeError("BMI group not found for all patients")
grp = _patient_groups()
assert pd.Series(grp).value_counts().to_dict() == {"overweight": 57, "normal_weight": 52, "obese": 18}, pd.Series(grp).value_counts()
XP = np.column_stack([grp == "overweight", grp == "obese"]).astype(float)
CONTRASTS = ["overweight_vs_normal", "obese_vs_normal"]

def simulate_grid(effect, rep):
    rng = np.random.default_rng([SEED, 1000 + GRID.index(effect), rep])
    m_c = np.full(n_c, effect); dev = rng.normal(0.0, T["dev_sd"][f2c]); beta = m_c[f2c] + dev
    shared = (T["int_sd"][None, :] * rng.standard_t(PATIENT_NU, size=(n_p, n_c)))[:, f2c]
    e = np.zeros((n_p, n_f))
    for j, (cols, L) in enumerate(CHOL):
        z = rng.standard_normal((n_p, len(cols))) @ L.T
        scale = np.sqrt(T["nu_c"][j] / rng.chisquare(T["nu_c"][j], size=(n_p, 1)))
        e[:, cols] = z * scale * T["sigma"][cols]
    grid = T["base"][None, :] + pcov[:, [0]] * beta[None, :] + pcov[:, 1:] @ T["cov_slope"] + shared + e
    obs = np.arange(len(F_IDX))
    return grid[P_IDX[obs], F_IDX[obs]], obs, beta, m_c

def dataset(scenario, rep):
    return simulate_grid(float(scenario.split("_")[1]), rep) if scenario.startswith("grid_") else simulate(scenario, rep)

def build_published(y, obs):
    fi, pi = F_IDX[obs], P_IDX[obs]
    K = np.unique(pi); ppos = np.searchsorted(K, pi); n = len(K)
    X = np.column_stack([np.ones(n), pcov[K]])
    u, s, _ = np.linalg.svd(X, full_matrices=True); r = int((s > max(X.shape) * np.finfo(float).eps * s.max()).sum())
    Q = u[:, r:] * np.sqrt(n / (n - r)); assert np.max(np.abs(X.T @ Q)) < 1e-9
    coords = {"feature": feats, "celltype": cts, "obs_id": np.arange(len(y)), "covariate": ["age_z", "sex_centered"],
              "patient_re_dimension": np.arange(Q.shape[1])}
    with pm.Model(coords=coords) as model:
        alpha = pm.Normal("feature_baseline", 0, .50, dims="feature")
        cm = pm.Normal("celltype_bmi_slope", 0, .20, dims="celltype")
        bs = pm.HalfNormal("feature_sigma_bmi", .30, dims="celltype")
        br = pm.Normal("feature_bmi_raw", 0, 1, dims="feature")
        beta = pm.Deterministic("feature_bmi_slope", cm[f2c] + bs[f2c] * br, dims="feature")
        cc = pm.Normal("celltype_covariate_slope", 0, .25, dims=("covariate", "celltype"))
        cs = pm.HalfNormal("feature_sigma_covariate", .25, dims=("covariate", "celltype"))
        cr = pm.Normal("feature_covariate_raw", 0, 1, dims=("covariate", "feature"))
        gamma = cc[:, f2c] + cs[:, f2c] * cr
        ps = pm.HalfNormal("patient_global_sigma", .50)
        pc = pm.Normal("patient_global_coefficient", 0, ps, dims="patient_re_dimension")
        pe = pt.dot(Q, pc)
        mu = alpha[fi] + beta[fi] * BMI_OBS[obs] + pe[ppos] + pt.sum(gamma[:, fi].T * COV_OBS[obs], axis=1)
        loc = pm.Normal("log_sigma_location", np.log(.65), .40)
        sc = pm.HalfNormal("log_sigma_celltype_location_sd", .25)
        rc = pm.Normal("log_sigma_celltype_raw", 0, 1, dims="celltype")
        sf = pm.HalfNormal("log_sigma_feature_scale", .50, dims="celltype")
        rf = pm.Normal("log_sigma_raw", 0, 1, dims="feature")
        sigma = pm.Deterministic("sigma_feature", pt.exp((loc + sc * rc)[f2c] + sf[f2c] * rf), dims="feature")
        if NU_SCOPE == "celltype":
            nu = pm.Deterministic("nu", 2 + pm.Exponential("nu_minus_two", .10, dims="celltype"), dims="celltype"); nu_o = nu[C_IDX[obs]]
        else:
            nu = pm.Deterministic("nu", 2 + pm.Exponential("nu_minus_two", .10)); nu_o = nu
        pm.StudentT("y_obs", nu=nu_o, mu=mu, sigma=sigma[fi], observed=y, dims="obs_id")
    return model

def build_group(y, obs):
    fi, pi, ci = F_IDX[obs], P_IDX[obs], C_IDX[obs]
    xg = XP[pi]
    coords = {"feature": feats, "celltype": cts, "patient": pats, "obs_id": np.arange(len(y)),
              "covariate": ["age_z", "sex_centered"], "contrast": CONTRASTS}
    with pm.Model(coords=coords) as model:
        base = pm.Normal("feature_baseline", 0.0, .50, dims="feature")
        cge = pm.Normal("celltype_group_effect", 0.0, CAT_CT_SD, dims=("contrast", "celltype"))
        fsg = pm.HalfNormal("feature_sigma_group", CAT_F_SD, dims=("contrast", "celltype"))
        fgr = pm.Normal("feature_group_raw", 0.0, 1.0, dims=("contrast", "feature"))
        dev = fgr * fsg[:, f2c]
        pm.Deterministic("feature_group_effect", cge[:, f2c] + dev, dims=("contrast", "feature"))
        ctc = pm.Normal("celltype_covariate_slope", 0.0, .25, dims=("covariate", "celltype"))
        fsc = pm.HalfNormal("feature_sigma_covariate", .25, dims=("covariate", "celltype"))
        fcr = pm.Normal("feature_covariate_raw", 0.0, 1.0, dims=("covariate", "feature"))
        cdev = fcr * fsc[:, f2c]
        isd = pm.HalfNormal("celltype_intercept_sigma", .50, dims="celltype")
        inu = pm.Deterministic("intercept_nu", pm.Exponential("intercept_nu_minus_two", lam=.10) + 2.0)
        wm = pt.dot(XP, cge) + pt.dot(pcov[:, 1:], ctc)
        w = pm.StudentT("celltype_intercept_w", nu=inu, mu=wm, sigma=isd[None, :], dims=("patient", "celltype"))
        mu = base[fi] + pt.sum(xg * dev[:, fi].T, axis=1) + w[pi, ci] + pt.sum(COV_OBS[obs] * cdev[:, fi].T, axis=1)
        loc = pm.Normal("log_sigma_location", np.log(.65), .40)
        sc = pm.HalfNormal("log_sigma_celltype_location_sd", .25)
        rc = pm.Normal("log_sigma_celltype_raw", 0, 1, dims="celltype")
        sf = pm.HalfNormal("log_sigma_feature_scale", .50, dims="celltype")
        rf = pm.Normal("log_sigma_raw", 0, 1, dims="feature")
        sigma = pm.Deterministic("sigma_feature", pt.exp((loc + sc * rc)[f2c] + sf[f2c] * rf), dims="feature")
        if NU_SCOPE == "celltype":
            nu = pm.Deterministic("nu", 2 + pm.Exponential("nu_minus_two", .10, dims="celltype"), dims="celltype"); nu_o = nu[ci]
        else:
            nu = pm.Deterministic("nu", 2 + pm.Exponential("nu_minus_two", .10)); nu_o = nu
        pm.StudentT("y_obs", nu=nu_o, mu=mu, sigma=sigma[fi], observed=y, dims="obs_id")
    return model

IDENT = {"published": ["feature_bmi_slope", "celltype_bmi_slope", "sigma_feature", "patient_global_sigma", "nu"],
         "group_t": ["feature_group_effect", "celltype_group_effect", "sigma_feature", "celltype_intercept_sigma", "intercept_nu", "nu"],
         "hc_t": ["feature_bmi_slope", "celltype_bmi_slope", "sigma_feature", "celltype_intercept_sigma", "intercept_nu", "nu"]}

def diag(idata, names):
    p, st = idata.posterior, idata.sample_stats
    r = max(float(np.nanmax(az_vals(az.rhat, p[v]))) for v in names)
    eb = min(float(np.nanmin(az_vals(az.ess, p[v], method="bulk"))) for v in names)
    et = min(float(np.nanmin(az_vals(az.ess, p[v], method="tail"))) for v in names)
    bfmi = float(np.min(az.bfmi(idata))); div = int(st["diverging"].values.sum())
    depth = float(st["maxdepth_reached"].values.mean() if "maxdepth_reached" in st else 0.0)
    ok = r < GATE["rhat"] and min(eb, et) >= GATE["ess"] and div == 0 and bfmi >= GATE["bfmi"] and depth <= GATE["maxdepth_fraction"]
    return dict(max_rhat=r, min_ess_bulk=eb, min_ess_tail=et, divergences=div, min_bfmi=bfmi, converged=bool(ok))

def sample(model, sampler, seed):
    t0 = time.time()
    with model:
        idata = pm.sample(tune=sampler["tune"], draws=sampler["draws"], chains=sampler["chains"], cores=sampler["chains"],
                          target_accept=sampler["target_accept"], nuts_sampler="nutpie", nuts_sampler_kwargs={"maxdepth": sampler["maxdepth"]},
                          random_seed=seed, return_inferencedata=True, progressbar=False)
    return idata, (time.time() - t0) / 60

def group_truth(beta, m_c, obs):
    K = np.unique(P_IDX[obs]); z = pcov[K, 0]; g = grp[K]
    d = {c: z[g == k].mean() - z[g == "normal_weight"].mean() for c, k in zip(CONTRASTS, ["overweight", "obese"])}
    return {c: (beta * d[c], m_c * d[c]) for c in CONTRASTS}

# ---------------- OLS per feature (HC3); BH within dataset
def ols(y, obs, design, coef_idx):
    fi, pi = F_IDX[obs], P_IDX[obs]
    res = {k: {"est": np.full(n_f, np.nan), "lo": np.full(n_f, np.nan), "hi": np.full(n_f, np.nan), "p": np.full(n_f, np.nan)} for k in coef_idx}
    for f in range(n_f):
        m = fi == f
        if m.sum() < 10: continue
        Xf, yy = design(pi[m]), y[m]
        XtXi = np.linalg.inv(Xf.T @ Xf); b = XtXi @ Xf.T @ yy; e = yy - Xf @ b; h = np.einsum("ij,jk,ik->i", Xf, XtXi, Xf)
        V = XtXi @ (Xf.T * (e / (1 - h)) ** 2) @ Xf @ XtXi; dfree = len(yy) - Xf.shape[1]; crit = stats.t.ppf(0.975, dfree)
        for name, j in coef_idx.items():
            se = np.sqrt(V[j, j]); res[name]["est"][f] = b[j]; res[name]["lo"][f] = b[j] - crit * se; res[name]["hi"][f] = b[j] + crit * se
            res[name]["p"][f] = 2 * stats.t.sf(abs(b[j] / se), dfree)
    return res

def ols_metrics(r, truth):
    ok = np.isfinite(r["p"]); q = np.full(n_f, np.nan); q[ok] = false_discovery_control(r["p"][ok], method="bh")
    null, nonnull = np.isclose(truth, 0) & ok, ~np.isclose(truth, 0) & ok
    right = np.sign(r["est"]) == np.sign(truth)
    out = {"coverage": float(((r["lo"] <= truth) & (truth <= r["hi"]))[ok].mean()), "mean_width": float(np.nanmean(r["hi"] - r["lo"])),
           "rmse": float(np.sqrt(np.nanmean((r["est"] - truth) ** 2))), "n_null": int(null.sum()), "n_nonnull": int(nonnull.sum())}
    for tag, call in [("unadj", ((r["lo"] > 0) | (r["hi"] < 0)) & ok), ("bh", (q < 0.05) & ok)]:
        out.update({f"{tag}_calls": int(call.sum()), f"{tag}_false": int((call & null).sum()), f"{tag}_true": int((call & nonnull & right).sum())})
    return out

# ---------------- tasks
def run_task(scenario, model, rep):
    out = OUT / f"{scenario}__{model}__r{rep:03d}.json"
    if out.exists(): return json.loads(out.read_text())
    y, obs, beta, m_c = dataset(scenario, rep)
    rec = {"scenario": scenario, "model": model, "replicate": rep, "base_config_hash": CONFIG_HASH,
           "data_sha256": hashlib.sha256(np.concatenate([y, obs.astype(float)]).tobytes()).hexdigest()}
    if model in ("ols", "ols_group"):
        if model == "ols":
            r = ols(y, obs, lambda p: np.column_stack([np.ones(len(p)), pcov[p]]), {"bmi": 1})
            rec["feature"] = ols_metrics(r["bmi"], beta)
        else:
            tr = group_truth(beta, m_c, obs)
            r = ols(y, obs, lambda p: np.column_stack([np.ones(len(p)), XP[p], pcov[p, 1:]]), {"overweight_vs_normal": 1, "obese_vs_normal": 2})
            rec["feature"] = {c: ols_metrics(r[c], tr[c][0]) for c in CONTRASTS}
        rec.update(converged_first=True, converged=True, minutes=0.0)
    else:
        attempts = []
        for attempt, sampler in enumerate([SAMPLER, RETRY_SAMPLER]):
            seed = int(hashlib.sha256(f"{scenario}|{model}|{rep}|{attempt}".encode()).hexdigest()[:8], 16)
            if model == "hc_t":
                idata, minutes = fit_once(0, y, obs, sampler, seed)
            else:
                idata, minutes = sample(build_published(y, obs) if model == "published" else build_group(y, obs), sampler, seed)
            d = diag(idata, IDENT[model]); p = idata.posterior
            a = {"attempt": attempt, "minutes": minutes, **d}
            if model == "group_t":
                tr = group_truth(beta, m_c, obs)
                a["feature"] = {c: interval_metrics(p["feature_group_effect"].sel(contrast=c), tr[c][0], "feature") for c in CONTRASTS}
                a["celltype"] = {c: interval_metrics(p["celltype_group_effect"].sel(contrast=c), tr[c][1], "celltype") for c in CONTRASTS}
            else:
                a["feature"] = interval_metrics(p["feature_bmi_slope"], beta, "feature")
                a["celltype"] = interval_metrics(p["celltype_bmi_slope"], m_c, "celltype")
            attempts.append(a); del idata, p; gc.collect()
            if d["converged"]: break
        rec.update(attempts=attempts, final_attempt=len(attempts) - 1, converged_first=attempts[0]["converged"],
                   converged=attempts[-1]["converged"], minutes=sum(x["minutes"] for x in attempts))
    tmp = out.with_suffix(".tmp"); tmp.write_text(json.dumps(rec, indent=2)); tmp.replace(out)
    return rec

TASKS = []
for r in range(REPLICATES):
    for s in SC6:
        TASKS += [(s, "published", r), (s, "group_t", r), (s, "ols_group", r)]
    for e in GRID:
        g = f"grid_{e:.2f}"
        TASKS += [(g, "hc_t", r), (g, "published", r), (g, "ols", r)]
mcmc = [t for t in TASKS if not t[1].startswith("ols")]
fast = [t for t in TASKS if t[1].startswith("ols")]
slot, nslot = SHARD * NWORKER + WORKER, NSHARD * NWORKER
MINE = [t for i, t in enumerate(mcmc) if i % nslot == slot] + ([t for t in fast] if slot == 0 else [])
print(f"sim2 shard {SHARD}/{NSHARD} worker {WORKER}/{NWORKER}: {len(MINE)} tasks of {len(TASKS)}", flush=True)
if RUN:
    for s, m, r in MINE:
        rec = run_task(s, m, r)
        if not m.startswith("ols"):
            a = rec["attempts"][rec["final_attempt"]]
            print(f"{s:16s} {m:10s} r{r:02d} attempts={len(rec['attempts'])} conv={rec['converged']} rhat={a['max_rhat']:.3f} "
                  f"div={a['divergences']} ({rec['minutes']:.1f} min)", flush=True)
    done = sum((OUT / f"{s}__{m}__r{r:03d}.json").exists() for s, m, r in MINE)
    print(f"sim2 shard {SHARD} worker {WORKER}: {done}/{len(MINE)} tasks finished", flush=True)


In [ ]:
import os, sys, subprocess, time
from pathlib import Path
SHARD_ID = 0
N_SHARDS = 3
N_WORKERS = max(1, (os.cpu_count() or 8) // 4)
OUT = Path("/content/drive/MyDrive/new_dysfunction/k1_validation_fine4_mount5_cpu_v1/tasks_v2")
LOG = OUT.parent / "logs_v2"; LOG.mkdir(parents=True, exist_ok=True)
print(f"{os.cpu_count()} CPUs -> {N_WORKERS} workers for shard {SHARD_ID}/{N_SHARDS}")
alive = lambda: [l for l in subprocess.run(["pgrep", "-fa", "sim2_worker.py"], capture_output=True, text=True).stdout.splitlines() if "pgrep" not in l]
if not alive():
    for w in range(N_WORKERS):
        env = dict(os.environ, SIM2_SHARD=str(SHARD_ID), SIM2_NSHARDS=str(N_SHARDS), SIM2_WORKER=str(w), SIM2_NWORKERS=str(N_WORKERS),
                   SIM2_RUN="1", OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
        f = open(LOG / f"shard{SHARD_ID}_w{w}.log", "a"); f.write(f"\n===== start {time.strftime('%Y-%m-%d %H:%M:%S')} =====\n"); f.flush()
        subprocess.Popen([sys.executable, "/content/sim2_worker.py"], stdout=f, stderr=subprocess.STDOUT, env=env, cwd="/content",
                         start_new_session=True)
        time.sleep(60)
else:
    print("workers already running here: monitoring only")
while True:
    print(time.strftime("%H:%M:%S"), f"| load {os.getloadavg()[0]:.1f} | task files {len(list(OUT.glob('*.json')))}")
    for lf in sorted(LOG.glob(f"shard{SHARD_ID}_w*.log")):
        lines = [l for l in lf.read_text(errors="ignore").splitlines() if any(k in l for k in ("attempts=", "finished", "Error", "Traceback"))]
        print(f"  {lf.stem}: {lines[-1][:150] if lines else '(starting)'}")
    if not alive():
        print("all workers in this runtime finished; when every shard is done, run cell 8"); break
    time.sleep(600)


In [ ]:
!pkill -f sim2_worker.py; echo stopped


In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path
from google.colab import files
SIM_ROOT = Path("/content/drive/MyDrive/new_dysfunction/k1_validation_fine4_mount5_cpu_v1")
SC6 = ["null_corr", "fitted_corr", "sparse_corr", "positive_copula", "contamination", "small_n"]
recs = [json.loads(p.read_text()) for p in sorted((SIM_ROOT / "tasks_v2").glob("*.json"))]
print(len(recs), "task files")
def ci(v):
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    return 1.96 * v.std(ddof=1) / np.sqrt(len(v)) if len(v) > 1 else np.nan
def mcmc_rows(model, scen_filter, level="feature", contrast=None):
    rows = []
    for r in recs:
        if r["model"] != model or not scen_filter(r["scenario"]): continue
        a = r["attempts"][r["final_attempt"]]; m = a[level][contrast] if contrast else a[level]
        rows.append(dict(scenario=r["scenario"], replicate=r["replicate"], converged=r["converged"], coverage=m["coverage"],
                         width=m["mean_width"], rmse=m["rmse"], false=m["false_calls"], n_null=m["n_null"], true=m["true_calls"],
                         n_nonnull=m["n_nonnull"], calls=m["n_calls"]))
    return pd.DataFrame(rows)
def ols_rows(model, scen_filter, tag, contrast=None):
    rows = []
    for r in recs:
        if r["model"] != model or not scen_filter(r["scenario"]): continue
        m = r["feature"][contrast] if contrast else r["feature"]
        rows.append(dict(scenario=r["scenario"], replicate=r["replicate"], converged=True, coverage=m["coverage"] if tag == "unadj" else np.nan,
                         width=m["mean_width"] if tag == "unadj" else np.nan, rmse=m["rmse"], false=m[f"{tag}_false"], n_null=m["n_null"],
                         true=m[f"{tag}_true"], n_nonnull=m["n_nonnull"], calls=m[f"{tag}_calls"]))
    return pd.DataFrame(rows)
def summarise(df, method, extra=None):
    out = []
    for s, g in df.groupby("scenario"):
        fdp = np.where(g.calls > 0, g["false"] / g.calls.replace(0, np.nan), 0.0)
        out.append(dict(scenario=s, method=method, **(extra or {}), n=len(g), converged=float(g.converged.mean()),
                        coverage=g.coverage.mean(), coverage_ci=ci(g.coverage), width=g.width.mean(), rmse=g.rmse.mean(),
                        fpr=g["false"].sum() / g.n_null.sum() if g.n_null.sum() else np.nan,
                        power=g["true"].sum() / g.n_nonnull.sum() if g.n_nonnull.sum() else np.nan, mean_fdp=float(np.mean(fdp))))
    return out
M = pd.read_csv(SIM_ROOT / "task_metrics_verified.csv"); H = M[(M.model == "hc_t") & M.scenario.isin(SC6)]
Hd = pd.DataFrame(dict(scenario=H.scenario, replicate=H.replicate, converged=H.converged_final, coverage=H.final_feature_coverage,
                       width=H.final_feature_mean_width, rmse=H.final_feature_rmse, false=H.final_feature_false_calls, n_null=H.final_feature_n_null,
                       true=H.final_feature_true_calls, n_nonnull=H.final_feature_n_nonnull, calls=H.final_feature_n_calls))
O = pd.read_csv(SIM_ROOT / "ols_feature_benchmark_by_replicate.csv")
Ou = O.rename(columns=dict(mean_width="width", unadj_false="false", unadj_true="true", unadj_calls="calls")).assign(converged=True)
Ob = O.rename(columns=dict(bh_false="false", bh_true="true", bh_calls="calls")).assign(converged=True, coverage=np.nan, mean_width=np.nan)
Ob["width"] = np.nan
in6 = lambda s: s in SC6
cont = pd.DataFrame(summarise(Hd, "Student-t") + summarise(mcmc_rows("published", in6), "Published model")
                    + summarise(Ou, "OLS") + summarise(Ob, "OLS + BH"))
isg = lambda s: s.startswith("grid_")
grid = pd.DataFrame(summarise(mcmc_rows("hc_t", isg), "Student-t") + summarise(mcmc_rows("published", isg), "Published model")
                    + summarise(ols_rows("ols", isg, "unadj"), "OLS") + summarise(ols_rows("ols", isg, "bh"), "OLS + BH"))
grid["effect"] = grid.scenario.str.replace("grid_", "").astype(float)
grp = []
for c in ["overweight_vs_normal", "obese_vs_normal"]:
    grp += summarise(mcmc_rows("group_t", in6, "feature", c), "Group model (Student-t), feature", {"contrast": c})
    grp += summarise(mcmc_rows("group_t", in6, "celltype", c), "Group model (Student-t), cell type", {"contrast": c})
    grp += summarise(ols_rows("ols_group", in6, "unadj", c), "OLS by group", {"contrast": c})
    grp += summarise(ols_rows("ols_group", in6, "bh", c), "OLS by group + BH", {"contrast": c})
grp = pd.DataFrame(grp)
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 20)
for name, t in [("sim2_continuous_summary.csv", cont), ("sim2_grid_summary.csv", grid), ("sim2_group_summary.csv", grp)]:
    t = t.round(4); t.to_csv(SIM_ROOT / name, index=False); print("\n" + name); print(t.to_string(index=False)); files.download(str(SIM_ROOT / name))


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]", "numpyro"])
import os, time, jax, numpy as np, pymc as pm
print("JAX devices:", jax.devices())
os.environ.update(SIM2_RUN="0")
W = {"__name__": "sim2_bench"}
exec(compile(open("/content/sim2_worker.py").read(), "sim2_worker.py", "exec"), W)
y, obs, beta, m_c = W["dataset"]("fitted_corr", 0)
S = W["SAMPLER"]
for name, build in [("published", W["build_published"]), ("group_t", W["build_group"])]:
    t0 = time.time()
    with build(y, obs):
        idata = pm.sample(tune=S["tune"], draws=S["draws"], chains=S["chains"], target_accept=S["target_accept"],
                          nuts_sampler="numpyro", nuts_sampler_kwargs={"chain_method": "vectorized", "max_tree_depth": S["maxdepth"]},
                          random_seed=1, progressbar=False)
    d = W["diag"](idata, W["IDENT"][name])
    print(f"{name}: {(time.time() - t0) / 60:.1f} min on GPU | conv={d['converged']} rhat={d['max_rhat']:.3f} div={d['divergences']}")
print("Compare with the CPU logs: about 9-15 min per fit, 2 fits at a time per 8-CPU runtime.")
